In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into persistent variables
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to add datetime column in UTC+8 (Asia/Shanghai)
def add_dt_col(df, ts_col='timestamp'):
    # ensure numeric
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    # create timezone-aware dt column
    df['ts_dt'] = pd.to_datetime(df[ts_col].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    return df

df_metric = add_dt_col(df_metric)
df_trace = add_dt_col(df_trace)
df_log = add_dt_col(df_log)
df_error = add_dt_col(df_error)

# Summarize metric.csv
metric_meta = pd.Series({
    'total_rows': len(df_metric),
    'min_timestamp': df_metric['ts_dt'].min() if len(df_metric) else pd.NaT,
    'max_timestamp': df_metric['ts_dt'].max() if len(df_metric) else pd.NaT
})

metric_top_cmdb = df_metric['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_top_kpi = df_metric['kpi_name'].value_counts().head(50).rename_axis('kpi_name').reset_index(name='count')
metric_top_combos = (df_metric.groupby(['cmdb_id','kpi_name']).size()
                     .reset_index(name='count')
                     .sort_values('count', ascending=False)
                     .head(20))
metric_samples = df_metric[['timestamp','ts_dt','cmdb_id','kpi_name','value']].head(5)

# Summarize trace.csv
trace_meta = pd.Series({
    'total_rows': len(df_trace),
    'min_timestamp': df_trace['ts_dt'].min() if len(df_trace) else pd.NaT,
    'max_timestamp': df_trace['ts_dt'].max() if len(df_trace) else pd.NaT
})
trace_top_cmdb = df_trace['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_top_name = df_trace['trace_name'].value_counts().head(50).rename_axis('trace_name').reset_index(name='count')
trace_top_combos = (df_trace.groupby(['cmdb_id','trace_name']).size()
                    .reset_index(name='count')
                    .sort_values('count', ascending=False)
                    .head(20))
trace_samples = df_trace[['timestamp','ts_dt','cmdb_id','trace_name','value']].head(5)

# Summarize log.csv
log_meta = pd.Series({
    'total_rows': len(df_log),
    'min_timestamp': df_log['ts_dt'].min() if len(df_log) else pd.NaT,
    'max_timestamp': df_log['ts_dt'].max() if len(df_log) else pd.NaT
})
log_top_cmdb = df_log['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_top_name = df_log['log_name'].value_counts().head(50).rename_axis('log_name').reset_index(name='count')
log_top_combos = (df_log.groupby(['cmdb_id','log_name']).size()
                  .reset_index(name='count')
                  .sort_values('count', ascending=False)
                  .head(20))
log_samples = df_log[['timestamp','ts_dt','cmdb_id','log_name','value']].head(5)

# Summarize error_logs.csv
error_meta = pd.Series({
    'total_rows': len(df_error),
    'min_timestamp': df_error['ts_dt'].min() if len(df_error) else pd.NaT,
    'max_timestamp': df_error['ts_dt'].max() if len(df_error) else pd.NaT
})
error_top_cmdb = df_error['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
error_samples = df_error[['timestamp','ts_dt','cmdb_id','message']].head(20)

# Compact outputs (do not print full loaded DataFrames; keep them for later use)
metric_meta, metric_top_cmdb, metric_top_kpi, metric_top_combos, metric_samples, \
trace_meta, trace_top_cmdb, trace_top_name, trace_top_combos, trace_samples, \
log_meta, log_top_cmdb, log_top_name, log_top_combos, log_samples, \
error_meta, error_top_cmdb, error_samples

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files (timestamps shown in UTC+8):\n\n"
    "1) metric.csv\n"
    "- Total rows: 319,703\n"
    "- Time range: 2022-03-20 12:30:00 to 2022-03-20 13:30:00\n"
    "- Top cmdb_id examples (top 10): adservice, adservice2, frontend2-0, frontend-2, frontend-1, frontend-0, "
    "checkoutservice-1, checkoutservice2-0, checkoutservice-2, checkoutservice-0\n"
    "- Notable KPI name patterns: many container.node-6.* metrics, app.grpc.*, container memory/cpu/fs metrics\n"
    "- Top (cmdb_id, kpi_name) combos: counts are small recurring blocks (top entries shown in results are ~61 records each)\n"
    "- Sample rows (up to 5): adservice rows such as app.grpc.count=250, app.grpc.mrt=0.002506, app.grpc.rr=100, app.grpc.sr=100, app.http.count=24\n\n"
    "2) trace.csv\n"
    "- Total rows: 35,768\n"
    "- Time range: 2022-03-20 13:00:00 to 2022-03-20 13:29:00\n"
    "- Top cmdb_id examples (top 10): frontend-2, frontend-0, checkoutservice-1, checkoutservice-2, checkoutservice-0, frontend-1, productcatalogservice-2, productcatalogservice-0, productcatalogservice-1, frontend2-0\n"
    "- Top trace_name examples: trace.from_frontend-2.row_count, trace.from_frontend-2.error_rate, trace.from_frontend-2.duration_p95, trace.from_frontend-2.duration_mean, and similar per-source/destination traces\n"
    "- Top (cmdb_id, trace_name) combos: common counts are ~30 per combo in the top list\n"
    "- Sample rows (up to 5): e.g., adservice-0 trace.from_frontend-0.duration_mean ~0.000018, duration_p95 ~0.000026, error_rate 0, row_count 28\n\n"
    "3) log.csv\n"
    "- Total rows: 1,838\n"
    "- Time range: 2022-03-20 13:00:00 to 2022-03-20 13:30:00\n"
    "- Top cmdb_id examples (top 10): adservice-2, cartservice-1, cartservice-0, cartservice-2, emailservice-1, currencyservice-0, currencyservice-1, currencyservice-2, productcatalogservice-0, frontend-2\n"
    "- Distinct log names (top): log.error_count and log.row_count (each with 919 occurrences overall)\n"
    "- Top (cmdb_id, log_name) combos: many combos appear ~31 times each in the top list\n"
    "- Sample rows (up to 5): e.g., adservice-0 log.error_count=0.0, adservice-0 log.row_count=132.0, etc.\n\n"
    "4) error_logs.csv\n"
    "- Total rows: 0 (file is empty / no error log rows)\n\n"
    "Notes:\n"
    "- All dataframes are loaded in memory as df_metric, df_trace, df_log, df_error for follow-up analysis.\n"
    "- If you want any deeper inspection (specific cmdb_id, KPI, time slices, or full top-N lists), tell me which filter and I will run concise queries."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(total_rows                          319703
min_timestamp    2022-03-20 12:30:00+08:00
max_timestamp    2022-03-20 13:30:00+08:00
dtype: object,                      cmdb_id  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  11956
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6          checkoutservice-1   8540
7         checkoutservice2-0   8540
8          checkoutservice-2   8296
9          checkoutservice-0   8296
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14  recommendationservice2-0   6222
15             cartservice-2   6100
16             cartservice-0   6100
17            cartservice2-0   6100
18   recommendationservice-1   5978
19   recommendationservice-2   5978,                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_receive_MB....   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_receive_err...   2196
..                                                ...    ...
45  container.node-6.container_memory_failures.hie...   2074
46  container.node-6.container_memory_failures.con...   2074
47  container.node-6.container_memory_failures.con...   2074
48        container.node-6.container_cpu_user_seconds   2074
49    container.node-6.container_fs_reads_MB./dev/vda   2074

[50 rows x 2 columns],                  cmdb_id                                           kpi_name  count
5252  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61
0              adservice                                     app.grpc.count     61
1              adservice                                       app.grpc.mrt     61
2              adservice                                        app.grpc.rr     61
3              adservice                                        app.grpc.sr     61
5213  shippingservice2-0  container.node-6.container_network_receive_MB....     61
5214  shippingservice2-0  container.node-6.container_network_receive_err...     61
5215  shippingservice2-0  container.node-6.container_network_receive_pac...     61
5216  shippingservice2-0  container.node-6.container_network_receive_pac...     61
5217  shippingservice2-0  container.node-6.container_network_transmit_MB...     61
5218  shippingservice2-0  container.node-6.container_network_transmit_er...     61
5219  shippingservice2-0  container.node-6.container_network_transmit_pa...     61
5220  shippingservice2-0  container.node-6.container_network_transmit_pa...     61
5205  shippingservice2-0  container.node-6.container_memory_failures.hie...     61
5206  shippingservice2-0  container.node-6.container_memory_failures.hie...     61
5207  shippingservice2-0      container.node-6.container_memory_mapped_file     61
5208  shippingservice2-0     container.node-6.container_memory_max_usage_MB     61
5209  shippingservice2-0              container.node-6.container_memory_rss     61
5210  shippingservice2-0             container.node-6.container_memory_swap     61
5211  shippingservice2-0         container.node-6.container_memory_usage_MB     61,     timestamp                     ts_dt    cmdb_id        kpi_name       value
0  1647750600 2022-03-20 12:30:00+08:00  adservice  app.grpc.count  250.000000
1  1647750600 2022-03-20 12:30:00+08:00  adservice    app.grpc.mrt    0.002506
2  1647750600 2022-03-20 12:30:00+08:00  adservice     app.grpc.rr  100.000000
3  1647750600 2022-03-20 12:30:00+08:00  adservice     app.grpc.sr  100.000000
4  1647750600 2022-03-20 12:30:00+08:00  adservice  app.http.count   24.000000, total_rows                           35768
min_timestamp    2022-03-20 13:00:00+08:00
max_timestamp    2022-03-20 13:29:00+08:00
dtype: object,                     cmdb_id  count
0                frontend-2   2760
1                frontend-0   2756
2         checkoutservice-1   2632
3         checkoutservice-2   2624
4         checkoutservice-0   2600
5                frontend-1   2288
6   productcatalogservice-2   1200
7   productcatalogservice-0   1200
8   productcatalogservice-1   1196
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-0    840
15            cartservice-2    840
16            cartservice-1    836
17        currencyservice-0    720
18        currencyservice-1    720
19        currencyservice-2    720,                               trace_name  count
0        trace.from_frontend-2.row_count    630
1       trace.from_frontend-2.error_rate    630
2     trace.from_frontend-2.duration_p95    630
3    trace.from_frontend-2.duration_mean    630
4    trace.from_frontend-0.duration_mean    629
..                                   ...    ...
45   trace.from_frontend2-0.duration_p95    210
46     trace.from_frontend2-0.error_rate    210
47      trace.from_frontend2-0.row_count    210
48  trace.to_cartservice-0.duration_mean    180
49   trace.to_cartservice-0.duration_p95    180

[50 rows x 2 columns],                  cmdb_id                           trace_name  count
1231  shippingservice2-0     trace.from_frontend2-0.row_count     30
0            adservice-0  trace.from_frontend-0.duration_mean     30
1            adservice-0   trace.from_frontend-0.duration_p95     30
2            adservice-0     trace.from_frontend-0.error_rate     30
3            adservice-0      trace.from_frontend-0.row_count     30
4            adservice-0  trace.from_frontend-1.duration_mean     30
5            adservice-0   trace.from_frontend-1.duration_p95     30
6            adservice-0     trace.from_frontend-1.error_rate     30
7            adservice-0      trace.from_frontend-1.row_count     30
8            adservice-0  trace.from_frontend-2.duration_mean     30
9            adservice-0   trace.from_frontend-2.duration_p95     30
10           adservice-0     trace.from_frontend-2.error_rate     30
11           adservice-0      trace.from_frontend-2.row_count     30
12           adservice-1  trace.from_frontend-0.duration_mean     30
13           adservice-1   trace.from_frontend-0.duration_p95     30
14           adservice-1     trace.from_frontend-0.error_rate     30
1196   shippingservice-1  trace.from_frontend-2.duration_mean     30
1197   shippingservice-1   trace.from_frontend-2.duration_p95     30
1198   shippingservice-1     trace.from_frontend-2.error_rate     30
1199   shippingservice-1      trace.from_frontend-2.row_count     30,     timestamp                     ts_dt      cmdb_id                           trace_name      value
0  1647752400 2022-03-20 13:00:00+08:00  adservice-0  trace.from_frontend-0.duration_mean   0.000018
1  1647752400 2022-03-20 13:00:00+08:00  adservice-0   trace.from_frontend-0.duration_p95   0.000026
2  1647752400 2022-03-20 13:00:00+08:00  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647752400 2022-03-20 13:00:00+08:00  adservice-0      trace.from_frontend-0.row_count  28.000000
4  1647752400 2022-03-20 13:00:00+08:00  adservice-0  trace.from_frontend-1.duration_mean   0.000016, total_rows                            1838
min_timestamp    2022-03-20 13:00:00+08:00
max_timestamp    2022-03-20 13:30:00+08:00
dtype: object,                     cmdb_id  count
0               adservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3             cartservice-2     62
4            emailservice-1     62
5         currencyservice-0     62
6         currencyservice-1     62
7         currencyservice-2     62
8   productcatalogservice-0     62
9                frontend-2     62
10  recommendationservice-1     62
11  recommendationservice-2     62
12               frontend-0     62
13              adservice-0     60
14        checkoutservice-0     60
15           emailservice-2     60
16           emailservice-0     60
17        checkoutservice-2     60
18        checkoutservice-1     60
19              adservice-1     60,           log_name  count
0  log.error_count    919
1    log.row_count    919,                     cmdb_id         log_name  count
4               adservice-2  log.error_count     31
5               adservice-2    log.row_count     31
9             cartservice-1    log.row_count     31
8             cartservice-1  log.error_count     31
7             cartservice-0    log.row_count     31
6             cartservice-0  log.error_count     31
10            cartservice-2  log.error_count     31
11            cartservice-2    log.row_count     31
27           emailservice-1    log.row_count     31
26           emailservice-1  log.error_count     31
18        currencyservice-0  log.error_count     31
19        currencyservice-0    log.row_count     31
20        currencyservice-1  log.error_count     31
21        currencyservice-1    log.row_count     31
23        currencyservice-2    log.row_count     31
22        currencyservice-2  log.error_count     31
43  productcatalogservice-0    log.row_count     31
42  productcatalogservice-0  log.error_count     31
34               frontend-2  log.error_count     31
35               frontend-2    log.row_count     31,     timestamp                     ts_dt      cmdb_id         log_name  value
0  1647752400 2022-03-20 13:00:00+08:00  adservice-0  log.error_count    0.0
1  1647752400 2022-03-20 13:00:00+08:00  adservice-0    log.row_count  132.0
2  1647752400 2022-03-20 13:00:00+08:00  adservice-1  log.error_count    0.0
3  1647752400 2022-03-20 13:00:00+08:00  adservice-1    log.row_count  118.0
4  1647752400 2022-03-20 13:00:00+08:00  adservice-2  log.error_count    0.0, total_rows         0
min_timestamp    NaT
max_timestamp    NaT
dtype: object, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, ts_dt, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing df_metric loaded in kernel
# ensure timezone and numeric value
tz = pytz.timezone('Asia/Shanghai')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
# ensure ts_dt exists and is timezone-aware (should already exist from prior step)
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95, P05, and total count per (cmdb_id, kpi_name)
grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
combo_stats = grp.agg(
    total_count='size',
    global_p95=lambda s: s.quantile(0.95),
    global_p05=lambda s: s.quantile(0.05)
).reset_index()

# 2) Select top 50 combos by total_count
top50_stats = combo_stats.sort_values('total_count', ascending=False).head(50).reset_index(drop=True)

# 3) For those top50 combos, filter their records to the failure window and compute windowed aggregates
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)

# restrict df_metric to only top50 combos (full series)
top50_keys = top50_stats[['cmdb_id', 'kpi_name']]
recs_top50 = df_metric.merge(top50_keys, on=['cmdb_id', 'kpi_name'], how='inner')

# create in-window mask
recs_top50['in_window'] = recs_top50['ts_dt'].between(start, end)

# merge global thresholds into recs_top50 to ease anomaly calc
recs_top50 = recs_top50.merge(top50_stats, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('', '_glob'))

# window subset
window_df = recs_top50[recs_top50['in_window']].copy()

# compute window-level aggregates per combo (handle combos with zero window rows later)
def calc_window_group(g):
    return pd.Series({
        'window_count': int(len(g)),
        'window_min': g['value'].min(),
        'window_max': g['value'].max(),
        'earliest_window_timestamp': g['ts_dt'].min(),
        'latest_window_timestamp': g['ts_dt'].max(),
        'anomaly_high_count': int((g['value'] > g['global_p95']).sum()),
        'anomaly_low_count': int((g['value'] < g['global_p05']).sum())
    })

if not window_df.empty:
    window_agg = window_df.groupby(['cmdb_id', 'kpi_name']).apply(calc_window_group).reset_index()
else:
    # empty placeholder
    window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count','window_min','window_max',
                                       'earliest_window_timestamp','latest_window_timestamp',
                                       'anomaly_high_count','anomaly_low_count'])

# 4) Combine top50 global stats with window_agg, fill missing window rows with zeros/NaT
final = top50_stats.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')

# fill missing window_count/anomaly counts with 0; keep min/max and timestamps as NaN/NaT
final['window_count'] = final['window_count'].fillna(0).astype(int)
final['anomaly_high_count'] = final['anomaly_high_count'].fillna(0).astype(int)
final['anomaly_low_count'] = final['anomaly_low_count'].fillna(0).astype(int)

# sort by total anomalies desc then total_count desc
final['total_anomalies'] = final['anomaly_high_count'] + final['anomaly_low_count']
final_sorted = final.sort_values(['total_anomalies','total_count'], ascending=[False, False])

# select and format columns for compact display (limit to top 20)
display_cols = [
    'cmdb_id','kpi_name','total_count','window_count',
    'global_p95','global_p05','window_min','window_max',
    'earliest_window_timestamp','latest_window_timestamp',
    'anomaly_high_count','anomaly_low_count'
]
final_table = final_sorted[display_cols].head(20)

# keep full intermediate results in memory for next steps:
# combo_stats, top50_stats, recs_top50, window_df, window_agg, final, final_sorted, final_table

final_table

```
Out[2]:
```
summary = (
    "Analysis summary (computed global P95/P05 per (cmdb_id,kpi); window = 2022-03-20 13:00–13:30 UTC+8):\n\n"
    "Key findings (top combos by anomaly count):\n"
    "1) adservice — app.grpc.mrt: total_count=61, window_count=31, global_p95≈0.002626, global_p05≈0.002437, "
    "window_min≈0.002402, window_max≈0.002669, earliest=2022-03-20 13:00, latest=13:30, anomaly_high=2, anomaly_low=2 (total 4)\n"
    "2) shippingservice2-0 — container.node-6.container_network_transmit_pa...: total_count=61, window_count=31, "
    "global_p95≈287.5, global_p05≈202.0, window_min≈199.0, window_max≈288.5, anomaly_high=1, anomaly_low=2 (total 3)\n"
    "3) shippingservice2-0 — container.node-6.container_last_seen: total_count=61, window_count=31, "
    "global_p95≈1.647754e9, global_p05≈1.647751e9, anomaly_high=3, anomaly_low=0 (total 3)\n"
    "4) shippingservice2-0 — container.node-6.container_fs_usage_MB./dev/vda1: total_count=61, window_count=31, "
    "global_p95≈50.281, global_p05≈49.830, anomaly_high=3, anomaly_low=0 (total 3)\n"
    "5) adservice — app.grpc.count: total_count=61, window_count=31, global_p95=260, global_p05=209, "
    "window_min=206, window_max=263, anomaly_high=1, anomaly_low=1 (total 2)\n"
    "6–8) Several shippingservice2-0 network receive/transmit metrics show anomaly totals of 1–2 (mixed high/low).\n"
    "Others in the top list have zero anomalies in the failure window.\n\n"
    "Observations:\n"
    "- The strongest signal is from adservice (app.grpc.mrt) with 4 total anomalies in the 13:00–13:30 window.\n"
    "- shippingservice2-0 shows multiple metrics (network, fs, last_seen) with elevated anomaly counts.\n"
    "- Most top combos have total_count=61 and window_count=31 (consistent sampling), so anomaly counts reflect real deviations vs global P95/P05.\n\n"
    "Recommendation: prioritize investigation of adservice (grpc latency metric) and shippingservice2-0 (network/fs/last_seen metrics) for root-cause clues. Full per-point data and traces/logs can be inspected next if you want.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_count  window_count    global_p95    global_p05    window_min    window_max earliest_window_timestamp   latest_window_timestamp  anomaly_high_count  anomaly_low_count
2            adservice                                       app.grpc.mrt           61            31  2.626349e-03  2.437445e-03  2.401702e-03  2.668584e-03 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   2                  2
11  shippingservice2-0  container.node-6.container_network_transmit_pa...           61            31  2.875000e+02  2.020000e+02  1.990000e+02  2.885000e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   1                  2
24  shippingservice2-0               container.node-6.container_last_seen           61            31  1.647754e+09  1.647751e+09  1.647752e+09  1.647754e+09 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   3                  0
34  shippingservice2-0   container.node-6.container_fs_usage_MB./dev/vda1           61            31  5.028125e+01  4.983008e+01  5.006641e+01  5.030273e+01 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   3                  0
1            adservice                                     app.grpc.count           61            31  2.600000e+02  2.090000e+02  2.060000e+02  2.630000e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   1                  1
5   shippingservice2-0  container.node-6.container_network_receive_MB....           61            31  3.231506e-01  3.147125e-02  3.125381e-02  3.248620e-01 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   1                  1
7   shippingservice2-0  container.node-6.container_network_receive_pac...           61            31  4.090000e+02  3.110000e+02  3.105000e+02  4.115000e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   1                  1
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61            31  2.039750e+03  1.044750e+03  9.950000e+02  2.039750e+03 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  1
9   shippingservice2-0  container.node-6.container_network_transmit_MB...           61            31  4.945226e-01  4.596357e-01  4.576340e-01  4.925647e-01 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  1
3            adservice                                        app.grpc.rr           61            31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
4            adservice                                        app.grpc.sr           61            31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
6   shippingservice2-0  container.node-6.container_network_receive_err...           61            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
8   shippingservice2-0  container.node-6.container_network_receive_pac...           61            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
10  shippingservice2-0  container.node-6.container_network_transmit_er...           61            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
12  shippingservice2-0  container.node-6.container_network_transmit_pa...           61            31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
13  shippingservice2-0  container.node-6.container_memory_failures.hie...           61            31  3.828000e+03  0.000000e+00  0.000000e+00  3.300000e+01 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
14  shippingservice2-0  container.node-6.container_memory_failures.hie...           61            31  1.650000e+01  0.000000e+00  0.000000e+00  0.000000e+00 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
15  shippingservice2-0      container.node-6.container_memory_mapped_file           61            31  2.297856e+06  8.110080e+05  8.110080e+05  8.110080e+05 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
16  shippingservice2-0     container.node-6.container_memory_max_usage_MB           61            31  1.287266e+02  1.287266e+02  1.287266e+02  1.287266e+02 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0
17  shippingservice2-0              container.node-6.container_memory_rss           61            31  9.678848e+06  8.396800e+06  8.667136e+06  8.937472e+06 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00                   0                  0```
```

In [ ]:
# In[3]:

# Reuse existing df_metric in kernel
import pandas as pd
# Aggregate total counts per (cmdb_id, kpi_name) across entire metric series
combo_counts = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False).size().reset_index(name='total_count')

# Filter for the two requested components and sort by count descending, limit to up to 100 rows each
ad_kpis = combo_counts[combo_counts['cmdb_id'] == 'adservice'].sort_values('total_count', ascending=False).head(100).reset_index(drop=True)
ship_kpis = combo_counts[combo_counts['cmdb_id'] == 'shippingservice2-0'].sort_values('total_count', ascending=False).head(100).reset_index(drop=True)

# Display compact results (two DataFrames)
ad_kpis, ship_kpis

```
Out[3]:
```
summary = (
    "Compact summary for requested components (based on df_metric aggregation):\n\n"
    "adservice:\n"
    "- Returned up to 100 distinct KPI names (100 shown). Many KPIs have total_count = 61 (full-series coverage).\n"
    "- Top example KPIs by count: runtime.jvm_threads_state.WAITING, app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr (each 61).\n\n"
    "shippingservice2-0:\n"
    "- Returned up to 80 distinct KPI names (80 shown). The majority of KPIs also have total_count = 61.\n"
    "- Top example KPIs by count: container.node-6.container_cpu_cfs_periods, container.node-6.container_cpu_cfs_throttled_*, container.node-6.container_cpu_load_average_10s, container.node-6.container_cpu_system_seconds, and many container.node-6.* and mesh.* metrics (each 61).\n\n"
    "Observation: For both components the telemetry is consistently sampled (most KPI series have 61 records), indicating full coverage across the captured timeframe. If you want the full lists or to export them as plain text, tell me which component and I will show more rows."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  total_count
0   adservice                  runtime.jvm_threads_state.WAITING           61
1   adservice                                     app.grpc.count           61
2   adservice                                       app.grpc.mrt           61
3   adservice                                        app.grpc.rr           61
4   adservice                                        app.grpc.sr           61
..        ...                                                ...          ...
95  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           61
96  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           61
97  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           61
98  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           61
99  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           61

[100 rows x 3 columns],                cmdb_id                                           kpi_name  total_count
0   shippingservice2-0         container.node-6.container_cpu_cfs_periods           61
1   shippingservice2-0  container.node-6.container_cpu_cfs_throttled_p...           61
2   shippingservice2-0  container.node-6.container_cpu_cfs_throttled_s...           61
3   shippingservice2-0    container.node-6.container_cpu_load_average_10s           61
4   shippingservice2-0      container.node-6.container_cpu_system_seconds           61
..                 ...                                                ...          ...
75  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...           61
76  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61
77  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61
78  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61
79  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61

[80 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric from kernel
tz = pytz.timezone('Asia/Shanghai')

# Target component and KPIs
component = 'adservice'
kpis = ["app.grpc.mrt","app.grpc.count","app.grpc.rr","app.grpc.sr"]

# Ensure numeric values and ts_dt present
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Filter series for component and KPIs (full series)
df_ad = df_metric[(df_metric['cmdb_id'] == component) & (df_metric['kpi_name'].isin(kpis))].copy()

# 1) Compute global P95, P05, total_count per KPI (using entire available series)
global_stats = (df_ad.groupby('kpi_name', dropna=False)['value']
                .agg(total_count='size',
                     global_p95=lambda s: s.quantile(0.95),
                     global_p05=lambda s: s.quantile(0.05))
                .reset_index())

# 2) Filter to failure window (inclusive) 2022-03-20 13:00:00 - 2022-03-20 13:30:00 (UTC+8)
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
df_ad['in_window'] = df_ad['ts_dt'].between(start, end)
df_ad_window = df_ad[df_ad['in_window']].copy()

# 3) For each KPI compute window_count and find anomaly points (value > global_p95 or < global_p05)
# Merge global thresholds into window df to evaluate anomalies
df_ad_window = df_ad_window.merge(global_stats, on='kpi_name', how='left', suffixes=('','_glob'))

# Identify anomalies
df_ad_window['anomaly_type'] = np.where(df_ad_window['value'] > df_ad_window['global_p95'], 'high',
                                        np.where(df_ad_window['value'] < df_ad_window['global_p05'], 'low', None))
anomalies_full = df_ad_window[df_ad_window['anomaly_type'].notna()].copy()
anomalies_full = anomalies_full.sort_values(['kpi_name','ts_dt'])

# Limit anomaly rows shown per KPI to 20 (but keep full in memory)
anomalies_limited = anomalies_full.groupby('kpi_name', group_keys=False).head(20)[
    ['kpi_name','ts_dt','value','anomaly_type']
].reset_index(drop=True)

# Compute window_count per KPI
window_counts = df_ad_window.groupby('kpi_name').size().reset_index(name='window_count')

# Compose summary table per KPI with global thresholds and counts
summary_df = global_stats.merge(window_counts, on='kpi_name', how='left')
summary_df['window_count'] = summary_df['window_count'].fillna(0).astype(int)

# 3.b Identify consecutive anomaly runs per KPI
runs_list = []
if not anomalies_full.empty:
    for kpi, grp in anomalies_full.groupby('kpi_name', sort=False):
        g = grp.sort_values('ts_dt').copy()
        # compute minute differences
        g['diff_min'] = g['ts_dt'].diff().dt.total_seconds().div(60).fillna(0)
        # new run when diff != 1
        g['run_id'] = (g['diff_min'] != 1).cumsum()
        # group into runs
        run_agg = (g.groupby('run_id', sort=False).agg(
            kpi_name=('kpi_name','first'),
            start_timestamp=('ts_dt','min'),
            end_timestamp=('ts_dt','max'),
            length_in_points=('ts_dt','size'),
            max_value_in_run=('value','max'),
            min_value_in_run=('value','min'),
            anomaly_types=('anomaly_type', lambda s: sorted(set(s)))
        ).reset_index(drop=True))
        # determine run-level anomaly_type and severity
        # fetch global thresholds for this kpi
        row = global_stats[global_stats['kpi_name']==kpi].iloc[0]
        gp95 = row['global_p95']
        gp05 = row['global_p05']
        def classify_and_severity(types, max_v, min_v):
            types_set = set(types)
            if types_set == {'high'}:
                typ = 'high'
                sev = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
                return typ, sev
            elif types_set == {'low'}:
                typ = 'low'
                sev = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
                return typ, sev
            else:
                typ = 'mixed'
                sev_high = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
                sev_low = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
                return typ, (sev_high, sev_low)
        # apply classification
        run_agg[['anomaly_type_run','severity']] = run_agg.apply(
            lambda r: pd.Series(classify_and_severity(r['anomaly_types'], r['max_value_in_run'], r['min_value_in_run'])),
            axis=1
        )
        # expand severity for mixed to readable numeric/text (keep as string for compactness)
        def fmt_sev(s):
            if isinstance(s, tuple):
                return f"high:{s[0]:.6f}, low:{s[1]:.6f}"
            elif pd.isna(s):
                return np.nan
            else:
                return f"{s:.6f}"
        run_agg['severity'] = run_agg['severity'].apply(fmt_sev)
        # select required columns and limit to first 10 runs per KPI
        run_agg = run_agg[['kpi_name','start_timestamp','end_timestamp','length_in_points',
                           'max_value_in_run','min_value_in_run','anomaly_type_run','severity']].head(10)
        runs_list.append(run_agg)
# concat runs (may be empty)
runs_full = pd.concat(runs_list, ignore_index=True) if runs_list else pd.DataFrame(columns=[
    'kpi_name','start_timestamp','end_timestamp','length_in_points',
    'max_value_in_run','min_value_in_run','anomaly_type_run','severity'
])

# Limit runs shown per KPI to first 10 (already handled). Create runs_limited for display
runs_limited = runs_full.copy()

# Keep full intermediate tables in variables:
# df_ad, df_ad_window, global_stats, anomalies_full, anomalies_limited, runs_full

# Final compact outputs (do not print raw full time series)
summary_df, anomalies_limited, runs_limited

```
Out[4]:
```
Analysis for adservice (window 2022-03-20 13:00–13:30 UTC+8)

Summary per KPI (global P95/P05 computed from entire series; window counts shown):
- app.grpc.count
  - total_count=61, window_count=31
  - global_p95=260, global_p05=209
  - window anomalies (≤20 shown): 
    - 2022-03-20 13:20:00 value=263 (high)
    - 2022-03-20 13:25:00 value=206 (low)
  - anomaly runs (<=10 shown):
    - 2022-03-20 13:20 → 13:20, length=1, max=263, min=263, type=high, severity=(263-260)/260 = 0.011538
    - 2022-03-20 13:25 → 13:25, length=1, max=206, min=206, type=low, severity=(209-206)/209 = 0.014354

- app.grpc.mrt
  - total_count=61, window_count=31
  - global_p95≈0.002626, global_p05≈0.002437
  - window anomalies (≤20 shown):
    - 2022-03-20 13:13:00 value≈0.002627 (high)
    - 2022-03-20 13:18:00 value≈0.002402 (low)
    - 2022-03-20 13:19:00 value≈0.002669 (high)
    - 2022-03-20 13:20:00 value≈0.002425 (low)
  - anomaly runs (<=10 shown):
    - 2022-03-20 13:13 → 13:13, length=1, max≈0.002627, min≈0.002627, type=high, severity=(max−p95)/p95 ≈ 0.000374
    - 2022-03-20 13:18 → 13:20, length=3, max≈0.002669, min≈0.002402, type=mixed,
      severity high ≈ (max−p95)/p95 = 0.016081, severity low ≈ (p05−min)/p05 = 0.014664

- app.grpc.rr
  - total_count=61, window_count=31
  - global_p95=100, global_p05=100
  - No anomalies detected in the window.

- app.grpc.sr
  - total_count=61, window_count=31
  - global_p95=100, global_p05=100
  - No anomalies detected in the window.

Key observations and recommendation:
- The strongest signal is in app.grpc.mrt: multiple anomaly points including a 3-minute consecutive run (13:18–13:20) containing both high and low anomalies (labeled "mixed") with modest relative severity (~1.6% above P95 / ~1.47% below P05).
- app.grpc.count shows two single-point anomalies (13:20 high, 13:25 low) with small relative severity (~1.15% and ~1.44%).
- app.grpc.rr and app.grpc.sr show no anomalies in the window.
- Suggest prioritizing investigation of adservice grpc latency (app.grpc.mrt), focusing on the 13:18–13:20 mixed run and adjacent points (13:13, 13:19–13:20) for potential transient causes.

The original code execution output of IPython Kernel is also provided below for reference:

(         kpi_name  total_count  global_p95  global_p05  window_count
0  app.grpc.count           61  260.000000  209.000000            31
1    app.grpc.mrt           61    0.002626    0.002437            31
2     app.grpc.rr           61  100.000000  100.000000            31
3     app.grpc.sr           61  100.000000  100.000000            31,          kpi_name                     ts_dt       value anomaly_type
0  app.grpc.count 2022-03-20 13:20:00+08:00  263.000000         high
1  app.grpc.count 2022-03-20 13:25:00+08:00  206.000000          low
2    app.grpc.mrt 2022-03-20 13:13:00+08:00    0.002627         high
3    app.grpc.mrt 2022-03-20 13:18:00+08:00    0.002402          low
4    app.grpc.mrt 2022-03-20 13:19:00+08:00    0.002669         high
5    app.grpc.mrt 2022-03-20 13:20:00+08:00    0.002425          low,          kpi_name           start_timestamp             end_timestamp  length_in_points  max_value_in_run  min_value_in_run anomaly_type_run                     severity
0  app.grpc.count 2022-03-20 13:20:00+08:00 2022-03-20 13:20:00+08:00                 1        263.000000        263.000000             high                     0.011538
1  app.grpc.count 2022-03-20 13:25:00+08:00 2022-03-20 13:25:00+08:00                 1        206.000000        206.000000              low                     0.014354
2    app.grpc.mrt 2022-03-20 13:13:00+08:00 2022-03-20 13:13:00+08:00                 1          0.002627          0.002627             high                     0.000374
3    app.grpc.mrt 2022-03-20 13:18:00+08:00 2022-03-20 13:20:00+08:00                 3          0.002669          0.002402            mixed  high:0.016081, low:0.014664)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure types and ts_dt exist
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Parameters
component = 'shippingservice2-0'
window_start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
window_end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)

# 1) Top 20 KPI names by total record count for the component
combo_counts = (df_metric[df_metric['cmdb_id'] == component]
                .groupby('kpi_name', dropna=False)['value']
                .size()
                .reset_index(name='total_count')
                .sort_values('total_count', ascending=False)
                .head(20)
                .reset_index(drop=True))

# 2) For those top20 KPIs compute global_p95 and global_p05 using the full series
top_kpis = combo_counts['kpi_name'].tolist()
full_top_series = df_metric[(df_metric['cmdb_id'] == component) & (df_metric['kpi_name'].isin(top_kpis))].copy()

global_thresh = (full_top_series.groupby('kpi_name')['value']
                 .agg(global_p95=lambda s: s.quantile(0.95),
                      global_p05=lambda s: s.quantile(0.05),
                      total_count='size')
                 .reset_index())

# merge counts to ensure consistency with combo_counts ordering
global_stats = combo_counts.merge(global_thresh, on=['kpi_name','total_count'], how='left')

# 3) For each KPI, compute window metrics and anomaly counts/timestamps
# filter window
window_df = full_top_series[full_top_series['ts_dt'].between(window_start, window_end)].copy()
# join thresholds to window records
window_df = window_df.merge(global_stats[['kpi_name','global_p95','global_p05']], on='kpi_name', how='left')

# identify anomalies
window_df['anomaly_type'] = np.where(window_df['value'] > window_df['global_p95'], 'high',
                                     np.where(window_df['value'] < window_df['global_p05'], 'low', None))

# aggregate per KPI
agg_window = (window_df.groupby('kpi_name', sort=False)
              .agg(window_count=('value','size'),
                   anomaly_high_count=('anomaly_type', lambda s: int((s=='high').sum())),
                   anomaly_low_count=('anomaly_type', lambda s: int((s=='low').sum())),
                   earliest_anomaly_timestamp=('ts_dt', lambda s: s[s.notna()].min() if any(s.notna()) else pd.NaT),
                   latest_anomaly_timestamp=('ts_dt', lambda s: s[s.notna()].max() if any(s.notna()) else pd.NaT))
              .reset_index())

# ensure KPIs with zero window rows are represented
agg_window = global_stats[['kpi_name','total_count','global_p95','global_p05']].merge(agg_window, on='kpi_name', how='left')
agg_window['window_count'] = agg_window['window_count'].fillna(0).astype(int)
agg_window['anomaly_high_count'] = agg_window['anomaly_high_count'].fillna(0).astype(int)
agg_window['anomaly_low_count'] = agg_window['anomaly_low_count'].fillna(0).astype(int)
agg_window['earliest_anomaly_timestamp'] = agg_window['earliest_anomaly_timestamp'].fillna(pd.NaT)
agg_window['latest_anomaly_timestamp'] = agg_window['latest_anomaly_timestamp'].fillna(pd.NaT)

# 4) Return compact table: top 10 KPIs sorted by anomaly total desc then total_count desc
agg_window['total_anomalies'] = agg_window['anomaly_high_count'] + agg_window['anomaly_low_count']
top10_table = agg_window.sort_values(['total_anomalies','total_count'], ascending=[False, False]) \
                       .loc[:, ['kpi_name','total_count','window_count','global_p95','global_p05',
                                'anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp','latest_anomaly_timestamp']] \
                       .head(10).reset_index(drop=True)

# 5) For the single KPI among these with largest total anomalies, list up to 20 anomaly points and identify consecutive runs
if not top10_table.empty:
    top_kpi = top10_table.iloc[0]['kpi_name']
    # get anomaly points in window for that KPI
    anomalies_kpi = window_df[window_df['kpi_name'] == top_kpi].copy()
    anomalies_kpi = anomalies_kpi[anomalies_kpi['anomaly_type'].notna()] \
                   .sort_values('ts_dt')[['ts_dt','value','anomaly_type']].reset_index(drop=True)
    anomalies_kpi_limited = anomalies_kpi.head(20).copy()
    # identify consecutive-minute runs
    if not anomalies_kpi.empty:
        a = anomalies_kpi.copy()
        a['diff_min'] = a['ts_dt'].diff().dt.total_seconds().div(60).fillna(0)
        a['run_id'] = (a['diff_min'] != 1).cumsum()
        runs = (a.groupby('run_id', sort=False).agg(
            start_timestamp=('ts_dt','min'),
            end_timestamp=('ts_dt','max'),
            length=('ts_dt','size'),
            max_value=('value','max'),
            min_value=('value','min'),
            anomaly_types=('anomaly_type', lambda s: sorted(set(s)))
        ).reset_index(drop=True))
        # classify run type and compute severity
        # fetch p95/p05 for this KPI
        row = agg_window[agg_window['kpi_name'] == top_kpi].iloc[0]
        gp95 = row['global_p95']
        gp05 = row['global_p05']
        def classify_and_sev(types, max_v, min_v):
            types_set = set(types)
            if types_set == {'high'}:
                typ = 'high'
                sev = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
                return typ, sev
            elif types_set == {'low'}:
                typ = 'low'
                sev = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
                return typ, sev
            else:
                typ = 'mixed'
                # compute both
                sev_high = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
                sev_low = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
                return typ, (sev_high, sev_low)
        runs[['anomaly_type_run','severity']] = runs.apply(
            lambda r: pd.Series(classify_and_sev(r['anomaly_types'], r['max_value'], r['min_value'])),
            axis=1
        )
        # format severity nicely
        def fmt_sev(s):
            if isinstance(s, tuple):
                return f"high:{s[0]:.6f}, low:{s[1]:.6f}"
            elif pd.isna(s):
                return np.nan
            else:
                return f"{s:.6f}"
        runs['severity'] = runs['severity'].apply(fmt_sev)
        runs = runs[['start_timestamp','end_timestamp','length','max_value','min_value','anomaly_type_run','severity']].head(10)
    else:
        anomalies_kpi_limited = anomalies_kpi.copy()
        runs = pd.DataFrame(columns=['start_timestamp','end_timestamp','length','max_value','min_value','anomaly_type_run','severity'])
else:
    top_kpi = None
    anomalies_kpi_limited = pd.DataFrame(columns=['ts_dt','value','anomaly_type'])
    runs = pd.DataFrame(columns=['start_timestamp','end_timestamp','length','max_value','min_value','anomaly_type_run','severity'])

# Keep full intermediate variables in memory for later: combo_counts, full_top_series, global_stats, window_df, agg_window, top10_table, anomalies_kpi, runs

# Compact display variables
top10_table, anomalies_kpi_limited, runs

```
Out[5]:
```
Summary for shippingservice2-0 (window 2022-03-20 13:00–13:30 UTC+8)

Top KPI patterns
- The top KPIs are dominated by container.node-6.* metrics (CPU, fs, file descriptors, inodes, etc.).
- For the top KPIs shown, total_count = 61 and window_count = 31 (consistent sampling across the timeframe).

Top 10 KPIs (compact view; shown sorted by anomaly total then count — columns: kpi_name, total_count, window_count, global_p95, global_p05, anomaly_high_count, anomaly_low_count)
- container.node-6.container_cpu_cfs_periods: total=61, window=31, p95=57.5, p05=21.0, high=0, low=2
- container.node-6.container_cpu_usage_seconds: total=61, window=31, p95≈0.358638, p05≈0.018716, high=0, low=1
- container.node-6.container_cpu_cfs_throttled_periods: total=61, window=31, p95=15.0, p05=0.0, high=0, low=0
- container.node-6.container_cpu_cfs_throttled_seconds: total=61, window=31, p95≈2.904714, p05=0.0, high=0, low=0
- container.node-6.container_cpu_load_average_10s: total=61, window=31, p95=0.0, p05=0.0, high=0, low=0
- container.node-6.container_cpu_system_seconds: total=61, window=31, p95=0.065, p05=0.0, high=0, low=0
- container.node-6.container_cpu_user_seconds: total=61, window=31, p95=0.285, p05=0.005, high=0, low=0
- container.node-6.container_file_descriptors: total=61, window=31, p95=15.0, p05=9.0, high=0, low=0
- container.node-6.container_fs_inodes./dev/vda1: total=61, window=31, p95=0.0, p05=0.0, high=0, low=0
- container.node-6.container_fs_inodes_free./dev/vda1: total=61, window=31, p95=0.0, p05=0.0, high=0, low=0

Top KPI with most anomalies
- KPI: container.node-6.container_cpu_cfs_periods
  - total_count = 61, window_count = 31
  - global_p95 = 57.5, global_p05 = 21.0
  - anomaly counts in window: anomaly_high_count = 0, anomaly_low_count = 2
  - anomaly points (up to 20):
    - 2022-03-20 13:13:00 value = 18.0 (low)
    - 2022-03-20 13:22:00 value = 19.5 (low)
  - consecutive anomaly runs (grouped by contiguous-minute timestamps):
    - Run 1: start=2022-03-20 13:13:00, end=2022-03-20 13:13:00, length=1, max=18.0, min=18.0, type=low, severity=(21.0−18.0)/21.0 = 0.142857
    - Run 2: start=2022-03-20 13:22:00, end=2022-03-20 13:22:00, length=1, max=19.5, min=19.5, type=low, severity=(21.0−19.5)/21.0 = 0.071429

Observations / next steps
- shippingservice2-0 shows a few low-value anomalies in container CPU CFS periods and one low in CPU usage; most other top KPIs show no anomalies in the failure window.
- The CPU CFS periods low-value points (13:13 and 13:22) have modest-to-small severity (14% and 7% relative to p05).
- Suggest investigating node-6/container resource scheduling or throttling around those timestamps and correlating with logs/traces for shippingservice2-0 and node-6. Full intermediate tables are retained in kernel variables for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name  total_count  window_count  global_p95  global_p05  anomaly_high_count  anomaly_low_count earliest_anomaly_timestamp  latest_anomaly_timestamp
0         container.node-6.container_cpu_cfs_periods           61            31   57.500000   21.000000                   0                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
1       container.node-6.container_cpu_usage_seconds           61            31    0.358638    0.018716                   0                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
2  container.node-6.container_cpu_cfs_throttled_p...           61            31   15.000000    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
3  container.node-6.container_cpu_cfs_throttled_s...           61            31    2.904714    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
4    container.node-6.container_cpu_load_average_10s           61            31    0.000000    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
5      container.node-6.container_cpu_system_seconds           61            31    0.065000    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
6        container.node-6.container_cpu_user_seconds           61            31    0.285000    0.005000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
7        container.node-6.container_file_descriptors           61            31   15.000000    9.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
8     container.node-6.container_fs_inodes./dev/vda1           61            31    0.000000    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
9  container.node-6.container_fs_inodes_free./dev...           61            31    0.000000    0.000000                   0                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00,                       ts_dt  value anomaly_type
0 2022-03-20 13:13:00+08:00   18.0          low
1 2022-03-20 13:22:00+08:00   19.5          low,             start_timestamp             end_timestamp  length  max_value  min_value anomaly_type_run  severity
0 2022-03-20 13:13:00+08:00 2022-03-20 13:13:00+08:00       1       18.0       18.0              low  0.142857
1 2022-03-20 13:22:00+08:00 2022-03-20 13:22:00+08:00       1       19.5       19.5              low  0.071429)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure value numeric and ts_dt exists
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter trace rows where cmdb_id contains 'adservice' OR cmdb_id == 'shippingservice2-0'
mask = df_trace['cmdb_id'].str.contains('adservice', na=False) | (df_trace['cmdb_id'] == 'shippingservice2-0')
df_trace_sel = df_trace[mask].copy()

# 2) For each (cmdb_id, trace_name) compute global_p95 and global_p05 across full series
grp = df_trace_sel.groupby(['cmdb_id','trace_name'], dropna=False)['value']
global_trace_stats = grp.agg(
    total_count='size',
    global_p95=lambda s: s.quantile(0.95),
    global_p05=lambda s: s.quantile(0.05)
).reset_index()

# 3) Filter to failure window and compute window metrics and anomalies
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
df_trace_sel['in_window'] = df_trace_sel['ts_dt'].between(start, end)
window_df = df_trace_sel[df_trace_sel['in_window']].copy()

# merge global thresholds into window_df
window_df = window_df.merge(global_trace_stats[['cmdb_id','trace_name','global_p95','global_p05','total_count']],
                            on=['cmdb_id','trace_name'], how='left')

# identify anomalies in window
window_df['anomaly_type'] = np.where(window_df['value'] > window_df['global_p95'], 'high',
                                     np.where(window_df['value'] < window_df['global_p05'], 'low', None))

# aggregate per (cmdb_id, trace_name)
agg_window = (window_df.groupby(['cmdb_id','trace_name'], sort=False).agg(
    window_count=('value','size'),
    anomaly_high_count=('anomaly_type', lambda s: int((s=='high').sum())),
    anomaly_low_count=('anomaly_type', lambda s: int((s=='low').sum())),
    earliest_anomaly_timestamp=('ts_dt', lambda s: s[s.notna()].min() if any(s.notna()) else pd.NaT),
    latest_anomaly_timestamp=('ts_dt', lambda s: s[s.notna()].max() if any(s.notna()) else pd.NaT)
).reset_index())

# 4) Ensure all combos from global_trace_stats are represented; merge on cmdb_id & trace_name
agg_all = global_trace_stats.merge(agg_window, on=['cmdb_id','trace_name'], how='left')

# fill missing window-related fields
agg_all['window_count'] = agg_all['window_count'].fillna(0).astype(int)
agg_all['anomaly_high_count'] = agg_all['anomaly_high_count'].fillna(0).astype(int)
agg_all['anomaly_low_count'] = agg_all['anomaly_low_count'].fillna(0).astype(int)
agg_all['earliest_anomaly_timestamp'] = agg_all['earliest_anomaly_timestamp'].fillna(pd.NaT)
agg_all['latest_anomaly_timestamp'] = agg_all['latest_anomaly_timestamp'].fillna(pd.NaT)

# prepare final table sorted by total anomalies then total_count, limit to top 20
agg_all['total_anomalies'] = agg_all['anomaly_high_count'] + agg_all['anomaly_low_count']
top20_table = (agg_all.sort_values(['total_anomalies','total_count'], ascending=[False, False])
                     .loc[:, ['cmdb_id','trace_name','total_count','window_count','global_p95','global_p05',
                              'anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp','latest_anomaly_timestamp']]
                     .head(20)
                     .reset_index(drop=True))

# 5) For the single (cmdb_id, trace_name) with largest total anomalies, list up to 20 anomaly points in the window
if not agg_all.empty:
    top_row = agg_all.sort_values(['total_anomalies','total_count'], ascending=[False, False]).iloc[0]
    top_cmdb = top_row['cmdb_id']
    top_trace = top_row['trace_name']
    anomalies_top = window_df[(window_df['cmdb_id']==top_cmdb) & (window_df['trace_name']==top_trace) & (window_df['anomaly_type'].notna())] \
                   .sort_values('ts_dt')[['ts_dt','value','anomaly_type']].reset_index(drop=True)
    anomalies_top_limited = anomalies_top.head(20).copy()
else:
    top_cmdb = None
    top_trace = None
    anomalies_top_limited = pd.DataFrame(columns=['ts_dt','value','anomaly_type'])

# Keep intermediate variables in kernel: df_trace_sel, global_trace_stats, window_df, agg_window, agg_all, top20_table, anomalies_top_limited

top20_table, anomalies_top_limited

```
Out[6]:
```
Summary of trace analysis for adservice* and shippingservice2-0 (window 2022-03-20 13:00–13:30 UTC+8)

Overall:
- We examined trace series where cmdb_id contains "adservice" or equals "shippingservice2-0".
- Each candidate (cmdb_id, trace_name) has full-series counts (total_count) of 30 and window_count = 30 (consistent per-minute sampling over the 30-minute window).
- Global thresholds (global P95 / P05) were computed from the entire available trace series, then used to detect window anomalies.

Top results (up to 20 rows sorted by total anomalies then count) — examples from the top of the table:
- adservice-0, trace.from_frontend-0.duration_p95: total_count=30, window_count=30, global_p95≈0.000029, global_p05≈0.000021, anomaly_high_count=2, anomaly_low_count=2
- adservice-0, trace.from_frontend-1.duration_p95: total_count=30, window_count=30, global_p95≈0.000025, global_p05≈0.000015, anomaly_high_count=2, anomaly_low_count=2
- adservice-1, trace.from_frontend-0.duration_p95: total_count=30, window_count=30, global_p95≈0.000031, global_p05≈0.000017, anomaly_high_count=2, anomaly_low_count=2
- adservice-2 and adservice2-0 entries also appear with similar patterns (many trace.*.duration_* and row_count series exhibiting 1–4 anomalies in the window).
- shippingservice2-0 entries with anomalies include:
  - trace.from_checkoutservice2-0.duration_mean: total_count=30, window_count=30, global_p95≈0.000073, global_p05≈0.000064, anomaly_high_count=2, anomaly_low_count=2
  - trace.from_frontend2-0.duration_mean: total_count=30, window_count=30, global_p95≈0.000087, global_p05≈0.000069, anomaly_high_count=2, anomaly_low_count=2
  - trace.from_frontend2-0.duration_p95 and trace.from_checkoutservice2-0.duration_p95 also show anomalies (1–2 high/low).

Top (single) trace with the largest total anomalies
- Selected series: adservice-0 — trace.from_frontend-0.duration_p95 (2 highs + 2 lows = 4 anomalies)
- Anomaly points in the failure window (up to 20 shown; these are the detected anomalies):
  - 2022-03-20 13:04:00 value=0.000020 (low)
  - 2022-03-20 13:11:00 value=0.000019 (low)
  - 2022-03-20 13:14:00 value=0.000029 (high)
  - 2022-03-20 13:19:00 value=0.000032 (high)

Interpretation and next steps
- Multiple adservice trace endpoints show both high and low deviations vs their global P95/P05 during the 13:00–13:30 window. shippingservice2-0 also shows several trace anomalies (duration_mean/p95) in the same window.
- The strongest single signal in this subset is adservice-0 trace.from_frontend-0.duration_p95 (4 anomalies). Other adservice instances and shippingservice2-0 traces show 1–4 anomalies each.
- Recommended next actions: inspect traces and request/response details around the anomaly timestamps (notably ~13:04, 13:11, 13:14, 13:19) for adservice-0, and correlate with the metric anomalies already identified (adservice grpc metrics and shippingservice2-0 container metrics) and any logs for root-cause clues.

Intermediate tables (df_trace_sel, global_trace_stats, window_df, agg_all, top20_table, anomalies_top_limited) are retained in kernel variables for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                   trace_name  total_count  window_count  global_p95  global_p05  anomaly_high_count  anomaly_low_count earliest_anomaly_timestamp  latest_anomaly_timestamp
0          adservice-0           trace.from_frontend-0.duration_p95           30            30    0.000029    0.000021                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
1          adservice-0           trace.from_frontend-1.duration_p95           30            30    0.000025    0.000015                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
2          adservice-1           trace.from_frontend-0.duration_p95           30            30    0.000031    0.000017                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
3          adservice-2           trace.from_frontend-0.duration_p95           30            30    0.000026    0.000015                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
4          adservice-2          trace.from_frontend-1.duration_mean           30            30    0.000016    0.000012                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
5          adservice-2           trace.from_frontend-1.duration_p95           30            30    0.000018    0.000013                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
6         adservice2-0          trace.from_frontend2-0.duration_p95           30            30    0.000032    0.000024                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
7         adservice2-0             trace.from_frontend2-0.row_count           30            30   54.550000   39.350000                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
8   shippingservice2-0  trace.from_checkoutservice2-0.duration_mean           30            30    0.000073    0.000064                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
9   shippingservice2-0         trace.from_frontend2-0.duration_mean           30            30    0.000087    0.000069                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
10         adservice-0              trace.from_frontend-0.row_count           30            30   35.550000   27.000000                   2                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
11         adservice-0          trace.from_frontend-1.duration_mean           30            30    0.000023    0.000014                   1                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
12         adservice-0           trace.from_frontend-2.duration_p95           30            30    0.000038    0.000021                   2                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
13         adservice-1              trace.from_frontend-0.row_count           30            30   35.550000   27.000000                   2                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
14         adservice-2              trace.from_frontend-2.row_count           30            30   30.550000   25.000000                   2                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
15  shippingservice2-0   trace.from_checkoutservice2-0.duration_p95           30            30    0.000086    0.000069                   1                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
16  shippingservice2-0          trace.from_frontend2-0.duration_p95           30            30    0.000150    0.000076                   2                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
17         adservice-0              trace.from_frontend-2.row_count           30            30   30.550000   25.000000                   2                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
18         adservice-1          trace.from_frontend-1.duration_mean           30            30    0.000023    0.000015                   1                  1  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00
19         adservice-1           trace.from_frontend-1.duration_p95           30            30    0.000028    0.000015                   2                  0  2022-03-20 13:00:00+08:00 2022-03-20 13:29:00+08:00,                       ts_dt     value anomaly_type
0 2022-03-20 13:04:00+08:00  0.000020          low
1 2022-03-20 13:11:00+08:00  0.000019          low
2 2022-03-20 13:14:00+08:00  0.000029         high
3 2022-03-20 13:19:00+08:00  0.000032         high)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse df_trace from kernel; ensure ts_dt exists and value numeric
tz = pytz.timezone('Asia/Shanghai')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter rows where cmdb_id contains 'adservice' OR cmdb_id == 'shippingservice2-0'
#    and trace_name contains 'from_' or 'to_' or substring 'shippingservice' or 'adservice'
mask_component = df_trace['cmdb_id'].str.contains('adservice', na=False) | (df_trace['cmdb_id'] == 'shippingservice2-0')
mask_trace = df_trace['trace_name'].str.contains(r'(from_|to_|shippingservice|adservice)', na=False)
trace_rel = df_trace[mask_component & mask_trace].copy()

# 2) Deduplicated compact list of (cmdb_id, trace_name) with counts, top 50 by count desc
dedup_counts = (trace_rel.groupby(['cmdb_id','trace_name'], dropna=False)
                .size()
                .reset_index(name='count')
                .sort_values('count', ascending=False)
                .head(50)
                .reset_index(drop=True))

# 3) Rows where trace_name indicates direct call between the two services (case-sensitive substrings)
direct_mask = trace_rel['trace_name'].str.contains(r'(from_frontend|to_shippingservice|from_shippingservice|to_adservice)', na=False)
df_direct = trace_rel[direct_mask].copy()

# Filter direct-call rows to failure window and return earliest 20 rows
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
df_direct_window = df_direct[df_direct['ts_dt'].between(start, end)].sort_values('ts_dt').head(20)
# Select requested columns: timestamp (original), ts_dt (UTC+8 readable), cmdb_id, trace_name, value
direct_window_rows = df_direct_window[['timestamp','ts_dt','cmdb_id','trace_name','value']].reset_index(drop=True)

# Keep intermediate variables in kernel for follow-up:
# trace_rel, dedup_counts, df_direct, df_direct_window, direct_window_rows

# Compact outputs: top 50 dedup counts and up to 20 direct-call rows in window
dedup_counts, direct_window_rows

```
Out[7]:
```
Summary of trace relationships between adservice instances and shippingservice2-0 (compact, UTC+8):

- Scope: rows where cmdb_id contains "adservice" or equals "shippingservice2-0", and trace_name contains "from_" or "to_" or the substrings "shippingservice"/"adservice".

Top (cmdb_id, trace_name) patterns (deduplicated, top entries by count):
- Many adservice instances (adservice-0, adservice-1, adservice-2, adservice2-0) show trace.from_frontend-X.* series (duration_mean, duration_p95, error_rate, row_count). Each of these (cmdb_id, trace_name) occurs ~30 times (consistent per-minute sampling).
- shippingservice2-0 shows trace.from_frontend2-0.* series (duration_mean, duration_p95, error_rate, row_count) with similar counts (~30).
- In total the deduplicated list returned 48 (cmdb_id, trace_name) entries (top 50 slice); most entries have count = 30.

Direct-call indicators in the failure window (2022-03-20 13:00–13:30 UTC+8):
- The returned direct-call sample (earliest 20 rows in window) shows many trace.from_frontend* rows at 13:00 for multiple adservice instances and for shippingservice2-0 (e.g., trace.from_frontend-0.duration_mean/p95/row_count for adservice-0/1/2 and trace.from_frontend2-0.* for shippingservice2-0).
- No explicit "to_shippingservice" or "to_adservice" direct-call trace names appeared in the earliest-window sample; the interaction evidence is primarily "from_frontend" traces indicating frontend → service calls.
- Example rows at 13:00 in the window:
  - adservice-0 / trace.from_frontend-0.duration_mean = 0.000018
  - adservice-2 / trace.from_frontend-0.row_count = 28.0
  - adservice2-0 / trace.from_frontend2-0.row_count = 54.0
  - shippingservice2-0 / trace.from_frontend2-0.duration_mean = 0.000070
  - shippingservice2-0 / trace.from_frontend2-0.row_count = 10.0

Interpretation / concise conclusion:
- Trace telemetry shows both adservice instances and shippingservice2-0 being called from frontends (frontend and frontend2). These frontend→service traces are abundant and consistently sampled.
- Evidence of a direct named trace like "to_shippingservice" or "to_adservice" was not present in the earliest-window sample; relationships are observed via "from_frontend" traces on each service.
- For follow-up, correlate the frontend→adservice and frontend2→shippingservice2-0 trace timestamps (e.g., 13:00 and the other anomaly timestamps you previously identified) with metric anomalies and logs to trace causal flow.  

Intermediate filtered tables are retained in kernel variables for further correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                               trace_name  count
0          adservice-0      trace.from_frontend-0.duration_mean     30
1          adservice-0       trace.from_frontend-0.duration_p95     30
2          adservice-0         trace.from_frontend-0.error_rate     30
3          adservice-0          trace.from_frontend-0.row_count     30
4          adservice-0      trace.from_frontend-1.duration_mean     30
..                 ...                                      ...    ...
43  shippingservice2-0  trace.from_checkoutservice2-0.row_count     30
44  shippingservice2-0     trace.from_frontend2-0.duration_mean     30
45  shippingservice2-0      trace.from_frontend2-0.duration_p95     30
46  shippingservice2-0        trace.from_frontend2-0.error_rate     30
47  shippingservice2-0         trace.from_frontend2-0.row_count     30

[48 rows x 3 columns],      timestamp                     ts_dt             cmdb_id                            trace_name      value
0   1647752400 2022-03-20 13:00:00+08:00         adservice-0   trace.from_frontend-0.duration_mean   0.000018
1   1647752400 2022-03-20 13:00:00+08:00         adservice-2   trace.from_frontend-0.duration_mean   0.000015
2   1647752400 2022-03-20 13:00:00+08:00         adservice-2    trace.from_frontend-0.duration_p95   0.000017
3   1647752400 2022-03-20 13:00:00+08:00         adservice-2      trace.from_frontend-0.error_rate   0.000000
4   1647752400 2022-03-20 13:00:00+08:00         adservice-2       trace.from_frontend-0.row_count  28.000000
5   1647752400 2022-03-20 13:00:00+08:00         adservice-2   trace.from_frontend-1.duration_mean   0.000014
6   1647752400 2022-03-20 13:00:00+08:00         adservice-2    trace.from_frontend-1.duration_p95   0.000015
7   1647752400 2022-03-20 13:00:00+08:00         adservice-2      trace.from_frontend-1.error_rate   0.000000
8   1647752400 2022-03-20 13:00:00+08:00         adservice-2       trace.from_frontend-1.row_count   4.000000
9   1647752400 2022-03-20 13:00:00+08:00         adservice-2    trace.from_frontend-2.duration_p95   0.000020
10  1647752400 2022-03-20 13:00:00+08:00         adservice-2      trace.from_frontend-2.error_rate   0.000000
11  1647752400 2022-03-20 13:00:00+08:00         adservice-2       trace.from_frontend-2.row_count  28.000000
12  1647752400 2022-03-20 13:00:00+08:00        adservice2-0  trace.from_frontend2-0.duration_mean   0.000020
13  1647752400 2022-03-20 13:00:00+08:00        adservice2-0   trace.from_frontend2-0.duration_p95   0.000027
14  1647752400 2022-03-20 13:00:00+08:00        adservice2-0     trace.from_frontend2-0.error_rate   0.000000
15  1647752400 2022-03-20 13:00:00+08:00        adservice2-0      trace.from_frontend2-0.row_count  54.000000
16  1647752400 2022-03-20 13:00:00+08:00  shippingservice2-0  trace.from_frontend2-0.duration_mean   0.000070
17  1647752400 2022-03-20 13:00:00+08:00  shippingservice2-0   trace.from_frontend2-0.duration_p95   0.000073
18  1647752400 2022-03-20 13:00:00+08:00  shippingservice2-0     trace.from_frontend2-0.error_rate   0.000000
19  1647752400 2022-03-20 13:00:00+08:00  shippingservice2-0      trace.from_frontend2-0.row_count  10.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse df_metric from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure necessary columns and types
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Extract node identifier from kpi_name: 'container.node-N' -> 'node-N', else 'no-node'
# Create a series-level node_id mapping (per kpi_name)
def extract_node(kpi):
    m = re.search(r'container\.node-(\d+)', str(kpi))
    return f"node-{m.group(1)}" if m else "no-node"

# Add node_id column to df_metric (cached for reuse)
df_metric['node_id'] = df_metric['kpi_name'].apply(extract_node)

# 2) For every unique (cmdb_id, kpi_name) compute global_p95 and global_p05 (full series)
series_stats = (df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
                .agg(total_count='size',
                     global_p95=lambda s: s.quantile(0.95),
                     global_p05=lambda s: s.quantile(0.05))
                .reset_index())

# Also bring node_id into series_stats (node derived from kpi_name)
# Since node extraction depends only on kpi_name, map via first occurrence
kpi_to_node = df_metric[['kpi_name','node_id']].drop_duplicates().set_index('kpi_name')['node_id']
series_stats['node_id'] = series_stats['kpi_name'].map(kpi_to_node)

# 3) Filter each series to the failure window and count anomaly points where value > global_p95 or < global_p05
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
window_df = df_metric[df_metric['ts_dt'].between(start, end)].copy()

# Merge global thresholds (computed from full series) into window_df
window_df = window_df.merge(series_stats[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')

# Identify anomalies in the window
window_df['anomaly_type'] = np.where(window_df['value'] > window_df['global_p95'], 'high',
                                     np.where(window_df['value'] < window_df['global_p05'], 'low', None))
anomalies_df = window_df[window_df['anomaly_type'].notna()].copy()

# 4) Aggregate anomaly counts per extracted node identifier
# number of distinct series (cmdb_id,kpi_name) with at least one anomaly in window
if not anomalies_df.empty:
    # distinct series with anomaly per node
    distinct_series = (anomalies_df.groupby('node_id')
                       .apply(lambda g: g[['cmdb_id','kpi_name']].drop_duplicates().shape[0])
                       .reset_index(name='distinct_series_with_anomaly'))
    # total anomaly points per node
    total_points = (anomalies_df.groupby('node_id').size().reset_index(name='total_anomaly_points'))
    # earliest and latest anomaly timestamps per node
    times = (anomalies_df.groupby('node_id').agg(
                earliest_anomaly_timestamp=('ts_dt','min'),
                latest_anomaly_timestamp=('ts_dt','max')
            ).reset_index())
    # combine
    node_agg = distinct_series.merge(total_points, on='node_id').merge(times, on='node_id')
else:
    # empty structure
    node_agg = pd.DataFrame(columns=['node_id','distinct_series_with_anomaly','total_anomaly_points',
                                     'earliest_anomaly_timestamp','latest_anomaly_timestamp'])

# 5) Compact table: top 10 node identifiers sorted by total_anomaly_points desc
node_summary_table = node_agg.sort_values('total_anomaly_points', ascending=False).head(10).reset_index(drop=True)

# 6) For top node_id, list up to 10 series that contributed most anomaly points with earliest anomaly timestamp
if not node_summary_table.empty:
    top_node = node_summary_table.iloc[0]['node_id']
    series_agg = (anomalies_df[anomalies_df['node_id']==top_node]
                  .groupby(['cmdb_id','kpi_name'], dropna=False)
                  .agg(series_anomaly_points=('anomaly_type','size'),
                       earliest_anomaly_timestamp=('ts_dt','min'))
                  .reset_index()
                  .sort_values('series_anomaly_points', ascending=False)
                  .head(10)
                  .reset_index(drop=True))
else:
    top_node = None
    series_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','series_anomaly_points','earliest_anomaly_timestamp'])

# Keep intermediate variables in kernel for follow-up:
# series_stats, window_df, anomalies_df, node_agg, node_summary_table, series_agg

# Final compact outputs (text-friendly DataFrames)
node_summary_table, series_agg

```
Out[8]:
```
Summary of node-level anomaly aggregation (window 2022-03-20 13:00–13:30 UTC+8):

- Node extraction:
  - KPI names containing "container.node-N" were mapped to node-N (N = 1..6). KPI names without that pattern were labeled "no-node".

- Aggregated anomaly counts (per node_id):
  - no-node: 971 distinct series had ≥1 anomaly, total anomaly points = 2,332. Earliest anomaly = 2022-03-20 13:00:00, latest = 2022-03-20 13:30:00.
  - node-6: 439 distinct series, total anomaly points = 1,123. Earliest = 13:00:00, latest = 13:30:00.
  - node-5: 90 distinct series, total anomaly points = 225. Earliest = 13:00:00, latest = 13:30:00.
  - (Table limited to top 10; these three are the top contributors.)

- Top contributing series for the top node_id (no-node) — up to 10 series with most anomaly points (series_anomaly_points) and earliest anomaly time:
  - shippingservice-1 / mesh.destination.frontend.shippingservice.istio... — 6 points, earliest 13:11:00
  - adservice-2 / mesh.destination.frontend.adservice.istio_request... — 6 points, earliest 13:08:00
  - shippingservice-0 / mesh.destination.frontend.shippingservice.istio... — 6 points, earliest 13:11:00
  - shippingservice-1 / mesh.source.shippingservice.jaeger-collector... — 6 points, earliest 13:12:00
  - shippingservice-1 / mesh.source.shippingservice.jaeger-collector... — 6 points, earliest 13:09:00
  - node-6 / system.fs.inodes.free — 6 points, earliest 13:00:00
  - node-6 / system.fs.inodes.total — 6 points, earliest 13:00:00
  - node-6 / system.disk.free — 6 points, earliest 13:00:00
  - shippingservice-2 / mesh.destination.frontend.shippingservice.istio... — 6 points, earliest 13:11:00
  - shippingservice-1 / mesh.source.shippingservice.jaeger-collector... — 6 points, earliest 13:09:00

Interpretation and recommended next steps (concise):
- The largest anomaly volume is from "no-node" KPIs — these are non-container KPIs (mesh, application-level metrics) and account for the majority of anomaly points. This indicates the problem surface includes service/mesh-level signals, not only host/container metrics.
- node-6 is the largest node-specific source of anomalies among nodes (1,123 anomaly points across many container metrics). node-5 also shows notable anomalies but smaller scale.
- Top anomalous series include mesh-related metrics for shippingservice and adservice and system/fs metrics tied to node-6 (system disk/inode metrics).
- Actionable next steps: correlate the "no-node" mesh/service anomalies and the node-6 container/fs anomalies with traces and logs (timestamps cluster within 13:00–13:30) to determine causality — e.g., mesh errors or service-level failures driving container-level resource symptoms, or vice versa. Full intermediate tables are retained in memory for follow-up correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(   node_id  distinct_series_with_anomaly  total_anomaly_points earliest_anomaly_timestamp  latest_anomaly_timestamp
0  no-node                           971                  2332  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
1   node-6                           439                  1123  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
2   node-5                            90                   225  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00,              cmdb_id                                           kpi_name  series_anomaly_points earliest_anomaly_timestamp
0  shippingservice-1  mesh.destination.frontend.shippingservice.isti...                      6  2022-03-20 13:11:00+08:00
1        adservice-2  mesh.destination.frontend.adservice.istio_requ...                      6  2022-03-20 13:08:00+08:00
2  shippingservice-0  mesh.destination.frontend.shippingservice.isti...                      6  2022-03-20 13:11:00+08:00
3  shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...                      6  2022-03-20 13:12:00+08:00
4  shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...                      6  2022-03-20 13:09:00+08:00
5             node-6                              system.fs.inodes.free                      6  2022-03-20 13:00:00+08:00
6             node-6                             system.fs.inodes.total                      6  2022-03-20 13:00:00+08:00
7             node-6                                   system.disk.free                      6  2022-03-20 13:00:00+08:00
8  shippingservice-2  mesh.destination.frontend.shippingservice.isti...                      6  2022-03-20 13:11:00+08:00
9  shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...                      6  2022-03-20 13:09:00+08:00)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_log from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure types and ts_dt exist
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
if 'ts_dt' not in df_log.columns:
    df_log['ts_dt'] = pd.to_datetime(df_log['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, log_name) using full series
log_series_stats = (df_log.groupby(['cmdb_id','log_name'], dropna=False)['value']
                    .agg(total_count='size',
                         global_p95=lambda s: s.quantile(0.95),
                         global_p05=lambda s: s.quantile(0.05))
                    .reset_index())

# 2) Filter to failure window and prepare windowed data
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
window_log_df = df_log[df_log['ts_dt'].between(start, end)].copy()

# Merge global thresholds into windowed records
window_log_df = window_log_df.merge(log_series_stats[['cmdb_id','log_name','global_p95','global_p05','total_count']],
                                    on=['cmdb_id','log_name'], how='left')

# 3) Identify anomalies in window
window_log_df['anomaly_type'] = np.where(window_log_df['value'] > window_log_df['global_p95'], 'high',
                                         np.where(window_log_df['value'] < window_log_df['global_p05'], 'low', None))
anomalies_log_df = window_log_df[window_log_df['anomaly_type'].notna()].copy()

# Aggregate per series: total_count (from global), window_count, anomaly counts, earliest/latest anomaly timestamps
if not window_log_df.empty:
    agg_window_logs = (window_log_df.groupby(['cmdb_id','log_name'], sort=False)
                       .agg(window_count=('value','size'))
                       .reset_index())
else:
    agg_window_logs = pd.DataFrame(columns=['cmdb_id','log_name','window_count'])

if not anomalies_log_df.empty:
    agg_anomalies = (anomalies_log_df.groupby(['cmdb_id','log_name'], sort=False).agg(
                        anomaly_high_count=('anomaly_type', lambda s: int((s=='high').sum())),
                        anomaly_low_count=('anomaly_type', lambda s: int((s=='low').sum())),
                        earliest_anomaly_timestamp=('ts_dt', 'min'),
                        latest_anomaly_timestamp=('ts_dt', 'max')
                    ).reset_index())
else:
    agg_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_high_count','anomaly_low_count',
                                          'earliest_anomaly_timestamp','latest_anomaly_timestamp'])

# Combine global stats with window aggregates and anomaly aggregates
agg_all_logs = log_series_stats.merge(agg_window_logs, on=['cmdb_id','log_name'], how='left') \
                               .merge(agg_anomalies, on=['cmdb_id','log_name'], how='left')

# Fill missing window/anomaly fields
agg_all_logs['window_count'] = agg_all_logs['window_count'].fillna(0).astype(int)
agg_all_logs['anomaly_high_count'] = agg_all_logs['anomaly_high_count'].fillna(0).astype(int)
agg_all_logs['anomaly_low_count'] = agg_all_logs['anomaly_low_count'].fillna(0).astype(int)
agg_all_logs['earliest_anomaly_timestamp'] = agg_all_logs['earliest_anomaly_timestamp'].fillna(pd.NaT)
agg_all_logs['latest_anomaly_timestamp'] = agg_all_logs['latest_anomaly_timestamp'].fillna(pd.NaT)

# 4) Prepare compact top-20 table sorted by total anomalies desc then total_count desc
agg_all_logs['total_anomalies'] = agg_all_logs['anomaly_high_count'] + agg_all_logs['anomaly_low_count']
top20_table = (agg_all_logs.sort_values(['total_anomalies','total_count'], ascending=[False, False])
                           .loc[:, ['cmdb_id','log_name','total_count','window_count','global_p95','global_p05',
                                    'anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp','latest_anomaly_timestamp']]
                           .head(20)
                           .reset_index(drop=True))

# 5) For the single series with largest total anomalies, list up to 20 anomaly points in the window
if not agg_all_logs.empty:
    top_row = agg_all_logs.sort_values(['total_anomalies','total_count'], ascending=[False, False]).iloc[0]
    top_cmdb = top_row['cmdb_id']
    top_logname = top_row['log_name']
    anomalies_top = anomalies_log_df[(anomalies_log_df['cmdb_id']==top_cmdb) & (anomalies_log_df['log_name']==top_logname)] \
                   .sort_values('ts_dt')[['timestamp','ts_dt','value','anomaly_type']].reset_index(drop=True)
    anomalies_top_limited = anomalies_top.head(20).copy()
else:
    anomalies_top_limited = pd.DataFrame(columns=['timestamp','ts_dt','value','anomaly_type'])

# Keep intermediate variables in kernel for follow-up: log_series_stats, window_log_df, anomalies_log_df, agg_all_logs

# Final compact outputs
top20_table, anomalies_top_limited

```
Out[9]:
```
Summary of log-level anomaly scan (window 2022-03-20 13:00–13:30 UTC+8)

What was done:
- Computed global P95/P05 per (cmdb_id, log_name) using the full log series, then detected window anomalies where values > P95 or < P05.

Top findings (compact):
- Many services show symmetric log.row_count anomalies (typically 2 high & 2 low in the window). Examples in the top 20:
  - adservice-2 / log.row_count — total_count=31, window_count=31, global_p95=137.00, global_p05=108.00, anomaly_high=2, anomaly_low=2 (earliest 13:12, latest 13:30)
  - cartservice-0 / log.row_count — total_count=31, window_count=31, p95≈1045.50, p05≈928.00, anomaly_high=2, anomaly_low=2
  - cartservice-1 / log.row_count, cartservice-2 / log.row_count — same pattern (2 high / 2 low)
  - currencyservice-0/1/2, frontend-0/1/2, productcatalogservice-0/1/2, recommendationservice-1/2, and others also appear with the same 2-high/2-low pattern
- Most top entries are log.row_count series sampled across the window (consistent full-window coverage).

Single top series (most total anomalies)
- adservice-2 / log.row_count (2 highs + 2 lows = 4 anomalies)
  - Detected anomaly points in window (up to 20 shown; these 4 are all):
    - 2022-03-20 13:12:00 value=138.0 (high)
    - 2022-03-20 13:17:00 value=106.0 (low)
    - 2022-03-20 13:18:00 value=138.0 (high)
    - 2022-03-20 13:30:00 value=22.0  (low)
  - global thresholds: P95=137.00, P05=108.00

Interpretation and next steps (concise):
- Log volume (row_count) anomalies are widespread across many services during 13:00–13:30, with the same pattern of two spikes above P95 and two dips below P05 for many series.
- adservice-2 is the top single log series with 4 anomalies; timestamps span 13:12–13:30 and should be correlated with metric and trace anomalies previously identified (e.g., adservice grpc latency and shippingservice2-0 metrics).
- Recommended follow-up: examine logs around the adservice-2 anomaly timestamps (13:12, 13:17, 13:18, 13:30), correlate with trace requests and the metric anomalies (adservice grpc and shippingservice2-0 container/mesh signals) to determine causality.

Intermediate aggregated tables and the anomaly point list are retained in kernel variables for further correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id       log_name  total_count  window_count  global_p95  global_p05  anomaly_high_count  anomaly_low_count earliest_anomaly_timestamp  latest_anomaly_timestamp
0               adservice-2  log.row_count           31            31      137.00      108.00                   2                  2  2022-03-20 13:12:00+08:00 2022-03-20 13:30:00+08:00
1             cartservice-0  log.row_count           31            31     1045.50      928.00                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
2             cartservice-1  log.row_count           31            31     1050.00      916.00                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
3             cartservice-2  log.row_count           31            31     1051.50      902.00                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
4         currencyservice-0  log.row_count           31            31     1004.00      710.00                   2                  2  2022-03-20 13:09:00+08:00 2022-03-20 13:30:00+08:00
5         currencyservice-1  log.row_count           31            31      989.50      711.00                   2                  2  2022-03-20 13:09:00+08:00 2022-03-20 13:30:00+08:00
6         currencyservice-2  log.row_count           31            31     1002.00      707.50                   2                  2  2022-03-20 13:11:00+08:00 2022-03-20 13:30:00+08:00
7                frontend-0  log.row_count           31            31     2524.00     2147.50                   2                  2  2022-03-20 13:02:00+08:00 2022-03-20 13:30:00+08:00
8                frontend-2  log.row_count           31            31     2278.50     1828.00                   2                  2  2022-03-20 13:04:00+08:00 2022-03-20 13:30:00+08:00
9   productcatalogservice-0  log.row_count           31            31     1212.50      933.50                   2                  2  2022-03-20 13:04:00+08:00 2022-03-20 13:30:00+08:00
10  recommendationservice-1  log.row_count           31            31      251.00      210.50                   2                  2  2022-03-20 13:12:00+08:00 2022-03-20 13:30:00+08:00
11  recommendationservice-2  log.row_count           31            31      252.50      205.50                   2                  2  2022-03-20 13:18:00+08:00 2022-03-20 13:30:00+08:00
12              adservice-0  log.row_count           30            30      149.10      124.90                   2                  2  2022-03-20 13:01:00+08:00 2022-03-20 13:17:00+08:00
13        checkoutservice-0  log.row_count           30            30      107.60       35.05                   2                  2  2022-03-20 13:04:00+08:00 2022-03-20 13:25:00+08:00
14        checkoutservice-2  log.row_count           30            30      103.75       35.90                   2                  2  2022-03-20 13:06:00+08:00 2022-03-20 13:15:00+08:00
15           emailservice-2  log.row_count           30            30       18.10        7.45                   2                  2  2022-03-20 13:09:00+08:00 2022-03-20 13:25:00+08:00
16               frontend-1  log.row_count           30            30      248.50      159.95                   2                  2  2022-03-20 13:04:00+08:00 2022-03-20 13:28:00+08:00
17         paymentservice-2  log.row_count           30            30       18.55        6.45                   2                  2  2022-03-20 13:07:00+08:00 2022-03-20 13:16:00+08:00
18  productcatalogservice-1  log.row_count           30            30     1185.20     1018.70                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:28:00+08:00
19  productcatalogservice-2  log.row_count           30            30     1185.30     1021.85                   2                  2  2022-03-20 13:00:00+08:00 2022-03-20 13:27:00+08:00,     timestamp                     ts_dt  value anomaly_type
0  1647753120 2022-03-20 13:12:00+08:00  138.0         high
1  1647753420 2022-03-20 13:17:00+08:00  106.0          low
2  1647753480 2022-03-20 13:18:00+08:00  138.0         high
3  1647754200 2022-03-20 13:30:00+08:00   22.0          low)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
import re

# Ensure df_metric is available and prepared
tz = pytz.timezone('Asia/Shanghai')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Global P95/P05 per (cmdb_id, kpi_name) (cache if not present)
if 'series_stats' not in globals():
    series_stats = (df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
                    .agg(total_count='size',
                         global_p95=lambda s: s.quantile(0.95),
                         global_p05=lambda s: s.quantile(0.05))
                    .reset_index())

# Window selection
start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
window_metric_df = df_metric[df_metric['ts_dt'].between(start, end)].copy()

# Merge global thresholds into window data
window_metric_df = window_metric_df.merge(
    series_stats[['cmdb_id','kpi_name','global_p95','global_p05']],
    on=['cmdb_id','kpi_name'], how='left'
)

# Identify anomaly points in window
window_metric_df['anomaly_type'] = np.where(window_metric_df['value'] > window_metric_df['global_p95'], 'high',
                                            np.where(window_metric_df['value'] < window_metric_df['global_p05'], 'low', None))
anomalies_df = window_metric_df[window_metric_df['anomaly_type'].notna()].copy()

# Helper to compute runs per series
def compute_runs(g):
    g = g.sort_values('ts_dt').copy()
    if g.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
            'max_value','min_value','run_type','run_severity'
        ])
    g['diff_min'] = g['ts_dt'].diff().dt.total_seconds().div(60).fillna(0)
    g['run_id'] = (g['diff_min'] != 1).cumsum()
    runs = (g.groupby('run_id', sort=False).agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        start_timestamp=('ts_dt','min'),
        end_timestamp=('ts_dt','max'),
        length_in_points=('ts_dt','size'),
        max_value=('value','max'),
        min_value=('value','min'),
        anomaly_types=('anomaly_type', lambda s: sorted(set(s))),
        global_p95=('global_p95','first'),
        global_p05=('global_p05','first')
    ).reset_index(drop=True))
    def classify_and_sev(row):
        types = set(row['anomaly_types'])
        gp95 = row['global_p95']
        gp05 = row['global_p05']
        max_v = row['max_value']
        min_v = row['min_value']
        if types == {'high'}:
            sev = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
            return 'high', float(sev)
        if types == {'low'}:
            sev = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
            return 'low', float(sev)
        # mixed
        sev_h = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
        sev_l = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
        # for mixed, store max of abs fractions for ranking but keep both available
        return 'mixed', (float(sev_h) if not pd.isna(sev_h) else np.nan, float(sev_l) if not pd.isna(sev_l) else np.nan)
    runs[['run_type','run_severity_raw']] = runs.apply(lambda r: pd.Series(classify_and_sev(r)), axis=1)
    # normalize numeric run_severity for ranking (single-sided max)
    def norm(x):
        if isinstance(x, tuple):
            vals = [v for v in x if (v is not None and not np.isnan(v))]
            return max(vals) if vals else np.nan
        else:
            return x
    runs['run_severity'] = runs['run_severity_raw'].apply(norm)
    # keep concise columns
    runs_out = runs[['cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
                     'max_value','min_value','run_type','run_severity','run_severity_raw']].copy()
    return runs_out

# Compute runs_all once (for all series with anomalies)
if not anomalies_df.empty:
    runs_frames = []
    for (cmdb,kpi), g in anomalies_df.groupby(['cmdb_id','kpi_name'], sort=False):
        runs_frames.append(compute_runs(g))
    runs_all = pd.concat(runs_frames, ignore_index=True) if runs_frames else pd.DataFrame(columns=[
        'cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
        'max_value','min_value','run_type','run_severity','run_severity_raw'
    ])
else:
    runs_all = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
        'max_value','min_value','run_type','run_severity','run_severity_raw'
    ])

# Define group membership masks on series_stats for consistent series listing
mask_adservice = series_stats['cmdb_id'].str.startswith('adservice', na=False)
mask_ship_pod = series_stats['cmdb_id'] == 'shippingservice2-0'
mask_node6 = series_stats['kpi_name'].str.contains('container.node-6', na=False)

groups_def = [
    ('adservice', mask_adservice),
    ('shippingservice2-0', mask_ship_pod),
    ('node-6', mask_node6)
]

group_summaries = []
group_top5 = {}

for gname, mask in groups_def:
    series_list = series_stats[mask][['cmdb_id','kpi_name']].drop_duplicates()
    if series_list.empty or runs_all.empty:
        distinct_series_with_runs = 0
        total_runs = 0
        total_anomaly_points = 0
        max_run_severity = np.nan
        earliest_run_start = pd.NaT
        latest_run_end = pd.NaT
        top5_df = pd.DataFrame(columns=['cmdb_id','kpi_name','series_max_run_severity','earliest_run_start'])
    else:
        # select runs that belong to this group's series
        runs_grp = runs_all.merge(series_list, on=['cmdb_id','kpi_name'], how='inner')
        if runs_grp.empty:
            distinct_series_with_runs = 0
            total_runs = 0
            total_anomaly_points = 0
            max_run_severity = np.nan
            earliest_run_start = pd.NaT
            latest_run_end = pd.NaT
            top5_df = pd.DataFrame(columns=['cmdb_id','kpi_name','series_max_run_severity','earliest_run_start'])
        else:
            distinct_series_with_runs = runs_grp[['cmdb_id','kpi_name']].drop_duplicates().shape[0]
            total_runs = int(runs_grp.shape[0])
            total_anomaly_points = int(runs_grp['length_in_points'].sum())
            max_run_severity = float(runs_grp['run_severity'].replace([np.inf,-np.inf], np.nan).dropna().max()) if runs_grp['run_severity'].notna().any() else np.nan
            earliest_run_start = runs_grp['start_timestamp'].min()
            latest_run_end = runs_grp['end_timestamp'].max()
            # per-series top severity
            per_series = (runs_grp.groupby(['cmdb_id','kpi_name'], as_index=False)
                          .agg(series_max_run_severity=('run_severity','max'),
                               earliest_run_start=('start_timestamp','min'))
                          .sort_values(['series_max_run_severity','earliest_run_start'], ascending=[False, True])
                          .head(5)
                          .reset_index(drop=True))
            top5_df = per_series[['cmdb_id','kpi_name','series_max_run_severity','earliest_run_start']]
    group_summaries.append({
        'group_name': gname,
        'distinct_series_with_runs': int(distinct_series_with_runs),
        'total_runs': int(total_runs),
        'total_anomaly_points': int(total_anomaly_points),
        'max_run_severity': max_run_severity,
        'earliest_run_start': earliest_run_start,
        'latest_run_end': latest_run_end
    })
    group_top5[gname] = top5_df

group_summary_df = pd.DataFrame(group_summaries)[[
    'group_name','distinct_series_with_runs','total_runs','total_anomaly_points',
    'max_run_severity','earliest_run_start','latest_run_end'
]]

# Final compact outputs: group_summary_df (3 rows) and top-5 per group
top5_ad = group_top5['adservice']
top5_ship = group_top5['shippingservice2-0']
top5_node6 = group_top5['node-6']

group_summary_df, top5_ad, top5_ship, top5_node6

```
Out[10]:
```
Summary (failure window 2022-03-20 13:00–13:30 UTC+8)

Group-level severity & timing (three rows)

- adservice
  - distinct_series_with_runs: 162
  - total_runs: 314
  - total_anomaly_points: 448
  - max_run_severity: 7.727934 (decimal fraction)
  - earliest_run_start: 2022-03-20 13:00:00+08:00
  - latest_run_end: 2022-03-20 13:30:00+08:00

- shippingservice2-0
  - distinct_series_with_runs: 19
  - total_runs: 25
  - total_anomaly_points: 30
  - max_run_severity: 1.928571
  - earliest_run_start: 2022-03-20 13:02:00+08:00
  - latest_run_end: 2022-03-20 13:30:00+08:00

- node-6
  - distinct_series_with_runs: 439
  - total_runs: 832
  - total_anomaly_points: 1123
  - max_run_severity: 24.046342
  - earliest_run_start: 2022-03-20 13:00:00+08:00
  - latest_run_end: 2022-03-20 13:30:00+08:00

Top contributing series per group (up to 5 each), ranked by their max run severity

- adservice (top 5)
  1) adservice2-0 — container.node-6.container_network_receive_MB... — series_max_run_severity = 7.727934 — earliest_run_start = 2022-03-20 13:02:00+08:00
  2) adservice-2 — container.node-5.container_network_receive_MB... — series_max_run_severity = 7.520692 — earliest_run_start = 2022-03-20 13:03:00+08:00
  3) adservice-1 — container.node-6.container_network_receive_MB... — series_max_run_severity = 4.065412 — earliest_run_start = 2022-03-20 13:05:00+08:00
  4) adservice-0 — container.node-6.container_network_receive_MB... — series_max_run_severity = 3.647863 — earliest_run_start = 2022-03-20 13:14:00+08:00
  5) adservice-2 — mesh.source.adservice.jaeger-collector... — series_max_run_severity = 1.000000 — earliest_run_start = 2022-03-20 13:05:00+08:00

- shippingservice2-0 (top 5)
  1) shippingservice2-0 — mesh.destination.frontend2.shippingservice2.is... — series_max_run_severity = 1.928571 — earliest_run_start = 2022-03-20 13:25:00+08:00
  2) shippingservice2-0 — mesh.source.shippingservice2.jaeger-collector... — series_max_run_severity = 0.338843 — earliest_run_start = 2022-03-20 13:06:00+08:00
  3) shippingservice2-0 — container.node-6.container_cpu_cfs_periods — series_max_run_severity = 0.142857 — earliest_run_start = 2022-03-20 13:13:00+08:00
  4) shippingservice2-0 — mesh.destination.frontend2.shippingservice2.is... — series_max_run_severity = 0.142857 — earliest_run_start = 2022-03-20 13:22:00+08:00
  5) shippingservice2-0 — mesh.destination.frontend2.shippingservice2.is... — series_max_run_severity = 0.138879 — earliest_run_start = 2022-03-20 13:22:00+08:00

- node-6 (top 5)
  1) paymentservice-0 — container.node-6.container_network_receive_MB... — series_max_run_severity = 24.046342 — earliest_run_start = 2022-03-20 13:11:00+08:00
  2) paymentservice-2 — container.node-6.container_network_receive_MB... — series_max_run_severity = 21.221708 — earliest_run_start = 2022-03-20 13:13:00+08:00
  3) shippingservice-0 — container.node-6.container_network_receive_MB... — series_max_run_severity = 11.511054 — earliest_run_start = 2022-03-20 13:18:00+08:00
  4) shippingservice-1 — container.node-6.container_network_receive_MB... — series_max_run_severity = 11.459349 — earliest_run_start = 2022-03-20 13:06:00+08:00
  5) paymentservice2-0 — container.node-6.container_memory_failures.con... — series_max_run_severity = 11.000000 — earliest_run_start = 2022-03-20 13:11:00+08:00

Concise interpretation and recommendation
- node-6 is the dominant hotspot by volume and severity (1,123 anomaly points, max run severity ~24x), driven by large network_receive and some memory-failure related runs on container.node-6 KPIs.
- adservice shows many affected series (162 series with runs) and high per-series severities — notably multiple adservice instances have very large network_receive MB run severities (7.7x, 7.5x, etc.), indicating strong anomalies tied to network I/O metrics for adservice containers.
- shippingservice2-0 has fewer runs and lower max severity than node-6/adservice, but has a notable mesh destination run (severity ~1.93) near 13:25.
- Prioritize investigation in this order: node-6 (node-level network/I/O anomalies), then adservice container network metrics and mesh metrics, then shippingservice2-0 mesh/run around 13:25. Correlate these series with traces and log anomalies (timestamps given) for root-cause confirmation.

Notes
- All thresholds (global P95/P05) were computed from full series as requested. Run-level details and intermediate tables are retained in kernel variables for follow-up correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(           group_name  distinct_series_with_runs  total_runs  total_anomaly_points  max_run_severity        earliest_run_start            latest_run_end
0           adservice                        162         314                   448          7.727934 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00
1  shippingservice2-0                         19          25                    30          1.928571 2022-03-20 13:02:00+08:00 2022-03-20 13:30:00+08:00
2              node-6                        439         832                  1123         24.046342 2022-03-20 13:00:00+08:00 2022-03-20 13:30:00+08:00,         cmdb_id                                           kpi_name  series_max_run_severity        earliest_run_start
0  adservice2-0  container.node-6.container_network_receive_MB....                 7.727934 2022-03-20 13:02:00+08:00
1   adservice-2  container.node-5.container_network_receive_MB....                 7.520692 2022-03-20 13:03:00+08:00
2   adservice-1  container.node-6.container_network_receive_MB....                 4.065412 2022-03-20 13:05:00+08:00
3   adservice-0  container.node-6.container_network_receive_MB....                 3.647863 2022-03-20 13:14:00+08:00
4   adservice-2  mesh.source.adservice.jaeger-collector.istio_r...                 1.000000 2022-03-20 13:05:00+08:00,               cmdb_id                                           kpi_name  series_max_run_severity        earliest_run_start
0  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                 1.928571 2022-03-20 13:25:00+08:00
1  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                 0.338843 2022-03-20 13:06:00+08:00
2  shippingservice2-0         container.node-6.container_cpu_cfs_periods                 0.142857 2022-03-20 13:13:00+08:00
3  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                 0.142857 2022-03-20 13:22:00+08:00
4  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                 0.138879 2022-03-20 13:22:00+08:00,              cmdb_id                                           kpi_name  series_max_run_severity        earliest_run_start
0   paymentservice-0  container.node-6.container_network_receive_MB....                24.046342 2022-03-20 13:11:00+08:00
1   paymentservice-2  container.node-6.container_network_receive_MB....                21.221708 2022-03-20 13:13:00+08:00
2  shippingservice-0  container.node-6.container_network_receive_MB....                11.511054 2022-03-20 13:18:00+08:00
3  shippingservice-1  container.node-6.container_network_receive_MB....                11.459349 2022-03-20 13:06:00+08:00
4  paymentservice2-0  container.node-6.container_memory_failures.con...                11.000000 2022-03-20 13:11:00+08:00)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric and ts_dt exist and value numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Ensure series_stats (global thresholds) exists; compute if missing
if 'series_stats' not in globals():
    series_stats = (df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
                    .agg(total_count='size',
                         global_p95=lambda s: s.quantile(0.95),
                         global_p05=lambda s: s.quantile(0.05))
                    .reset_index())

# Ensure node_id mapping exists on df_metric; compute if missing
if 'node_id' not in df_metric.columns:
    def extract_node(kpi):
        m = re.search(r'container\.node-(\d+)', str(kpi))
        return f"node-{m.group(1)}" if m else "no-node"
    df_metric['node_id'] = df_metric['kpi_name'].apply(extract_node)

# If runs_all not present, (re)compute anomaly runs using global thresholds
def compute_runs_for_anomalies(anom_df):
    # anom_df: anomalies for one series (already in-window and marked anomalous)
    g = anom_df.sort_values('ts_dt').copy()
    if g.empty:
        return pd.DataFrame()
    g['diff_min'] = g['ts_dt'].diff().dt.total_seconds().div(60).fillna(0)
    g['run_id'] = (g['diff_min'] != 1).cumsum()
    runs = (g.groupby('run_id', sort=False).agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        start_timestamp=('ts_dt','min'),
        end_timestamp=('ts_dt','max'),
        length_in_points=('ts_dt','size'),
        max_value=('value','max'),
        min_value=('value','min'),
        anomaly_types=('anomaly_type', lambda s: sorted(set(s))),
        global_p95=('global_p95','first'),
        global_p05=('global_p05','first')
    ).reset_index(drop=True))
    def classify_and_sev(row):
        types = set(row['anomaly_types'])
        gp95 = row['global_p95']
        gp05 = row['global_p05']
        max_v = row['max_value']
        min_v = row['min_value']
        if types == {'high'}:
            sev = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
            return 'high', float(sev)
        if types == {'low'}:
            sev = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
            return 'low', float(sev)
        # mixed
        sev_h = (max_v - gp95) / gp95 if gp95 not in (0, np.nan) else np.nan
        sev_l = (gp05 - min_v) / gp05 if gp05 not in (0, np.nan) else np.nan
        return 'mixed', (float(sev_h) if not pd.isna(sev_h) else np.nan, float(sev_l) if not pd.isna(sev_l) else np.nan)
    if not runs.empty:
        runs[['run_type','run_severity_raw']] = runs.apply(lambda r: pd.Series(classify_and_sev(r)), axis=1)
        # normalize numeric run_severity for filtering (max side for mixed)
        def norm(x):
            if isinstance(x, tuple):
                vals = [v for v in x if (v is not None and not np.isnan(v))]
                return max(vals) if vals else np.nan
            else:
                return x
        runs['run_severity'] = runs['run_severity_raw'].apply(norm)
    else:
        runs['run_type'] = []
        runs['run_severity_raw'] = []
        runs['run_severity'] = []
    return runs

if 'runs_all' not in globals():
    # Window selection
    start = pd.Timestamp('2022-03-20 13:00:00', tz=tz)
    end = pd.Timestamp('2022-03-20 13:30:00', tz=tz)
    window_metric_df = df_metric[df_metric['ts_dt'].between(start, end)].copy()
    # Merge global thresholds into window data
    window_metric_df = window_metric_df.merge(
        series_stats[['cmdb_id','kpi_name','global_p95','global_p05']],
        on=['cmdb_id','kpi_name'], how='left'
    )
    # Identify anomalies in the window
    window_metric_df['anomaly_type'] = np.where(window_metric_df['value'] > window_metric_df['global_p95'], 'high',
                                                np.where(window_metric_df['value'] < window_metric_df['global_p05'], 'low', None))
    anomalies_df = window_metric_df[window_metric_df['anomaly_type'].notna()].copy()
    # compute runs_all by grouping per series
    runs_frames = []
    for (cmdb,kpi), g in anomalies_df.groupby(['cmdb_id','kpi_name'], sort=False):
        runs_frames.append(compute_runs_for_anomalies(g))
    runs_all = pd.concat(runs_frames, ignore_index=True) if runs_frames else pd.DataFrame(columns=[
        'cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
        'max_value','min_value','anomaly_types','global_p95','global_p05','run_type','run_severity_raw','run_severity'
    ])

# Now filter runs_all for node-6 series: kpi_name contains 'container.node-6' OR series associated node_id == 'node-6'
# Map kpi_name to node_id via df_metric distinct mapping
kpi_to_node = df_metric[['kpi_name','node_id']].drop_duplicates().set_index('kpi_name')['node_id']
runs_all['node_id'] = runs_all['kpi_name'].map(kpi_to_node).fillna(runs_all['kpi_name'].apply(lambda s: ('node-6' if 'container.node-6' in str(s) else 'no-node')))
# Filter for node-6 association
runs_node6 = runs_all[(runs_all['kpi_name'].str.contains('container.node-6', na=False)) | (runs_all['node_id']=='node-6')].copy()

# Filter severity > 0.5
runs_node6_severe = runs_node6[runs_node6['run_severity'] > 0.5].copy()

# Merge global thresholds from series_stats to ensure columns present
runs_node6_severe = runs_node6_severe.merge(series_stats[['cmdb_id','kpi_name','global_p95','global_p05']],
                                            on=['cmdb_id','kpi_name'], how='left', suffixes=('','_glob'))

# Prepare final compact table with requested columns and renaming
final_cols = runs_node6_severe[['cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_points',
                               'run_type','run_severity','global_p95','global_p05','max_value','min_value']].copy()
final_cols = final_cols.rename(columns={
    'start_timestamp':'run_start_timestamp',
    'end_timestamp':'run_end_timestamp',
    'length_in_points':'run_length_in_points',
    'run_type':'run_type',
    'run_severity':'run_severity',
    'max_value':'max_value_in_run',
    'min_value':'min_value_in_run'
})

# Sort by run_start_timestamp ascending and limit to top 20 rows
node6_severe_runs_sorted = final_cols.sort_values('run_start_timestamp', ascending=True).head(20).reset_index(drop=True)

# Keep full results in kernel variables for follow-up
node6_severe_runs_all = runs_node6_severe
node6_severe_runs_head = node6_severe_runs_sorted

node6_severe_runs_head

```
Out[11]:
```
Compact summary of node-6 severe runs (run_severity > 0.5) — window 2022-03-20 13:00–13:30 UTC+8

What I extracted (top 20 runs by run start ascending):
- 20 severe runs on node-6-related KPI series (table shown in kernel). Runs start between 13:04 and 13:11 and extend through the 13:00–13:30 window.
- Most runs are short (1–2 minutes) and are dominated by container.node-6.network_receive MB and related network metrics; there are also CPU throttling and packet/transmit runs, and some mixed-type runs.

Notable example runs (from the top-20 list):
- 13:04 (redis-cart-0) — container.node-6.container_network_receive_MB..., length=1, type=high, severity≈4.84, max=0.66553
- 13:04–13:05 (cartservice-2) — container_network_receive_MB..., length=2, type=high, severity≈2.82, max≈0.3793
- 13:04–13:05 (emailservice-2) — container_network_receive_MB..., length=2, type=high, severity≈9.03, max≈0.3195
- 13:06 (shippingservice-1) — container_network_receive_MB..., length=1, type=high, severity≈11.46, max≈0.6219
- 13:09–13:10 (emailservice-0) — container_network_receive_MB..., length=2, type=high, severity≈8.52, max≈0.3189
- 13:09–13:11 (paymentservice-0) — container_network_receive_pac... (packets), length=1, type=high, severity≈0.566 (packets scaled), max=453.5
- 13:09–13:11 (paymentservice-0) — container_network_transmit_pa..., length=1, type=high, severity≈0.720, max=341.5
- 13:09–13:10 (checkoutservice-0) — container_network_receive_MB..., length=2, type=mixed, severity≈8.33 (high side), min_value also indicates some low-side breach in the mixed run

Overall interpretation and recommendations (concise)
- Node-6 shows concentrated, high-severity anomalies in container network receive/packet metrics across many services (adservice, paymentservice, shippingservice, emailservice, checkoutservice, cartservice, etc.). Several runs exceed 2×–11× the P95 baseline, indicating large relative spikes in network receive throughput or packet metrics on node-6.
- The timing is clustered in the 13:04–13:11 window (with additional related anomalies later), suggesting a short-lived but widespread network I/O event on node-6.
- Actionable next steps:
  1) Inspect node-6 network interface and host-level logs for errors/throttling around 13:04–13:12 (e.g., interface drops, NIC errors, interrupts, host-level throttling).
  2) Correlate these run timestamps with trace anomalies and the metric/log anomalies already identified for adservice and shippingservice2-0 (timestamps overlap).
  3) Check container scheduling/throttling and node disk/IO metrics as a secondary check (some mixed runs indicate combined effects).
- Full run records and the top-20 table are cached in kernel variables (node6_severe_runs_all, node6_severe_runs_head) for follow-up investigation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length_in_points run_type  run_severity  global_p95  global_p05  max_value_in_run  min_value_in_run
0              redis-cart-0  container.node-6.container_network_receive_MB.... 2022-03-20 13:04:00+08:00 2022-03-20 13:04:00+08:00                     1     high      4.838770    0.113985    0.087276          0.665530          0.665530
1             cartservice-2  container.node-6.container_network_receive_MB.... 2022-03-20 13:04:00+08:00 2022-03-20 13:05:00+08:00                     2     high      2.820512    0.099284    0.072994          0.379316          0.360796
2            emailservice-2  container.node-6.container_network_receive_MB.... 2022-03-20 13:04:00+08:00 2022-03-20 13:05:00+08:00                     2     high      9.030509    0.031853    0.022823          0.319499          0.316648
3        currencyservice2-0  container.node-6.container_network_receive_MB.... 2022-03-20 13:04:00+08:00 2022-03-20 13:04:00+08:00                     1     high      3.483201    0.156981    0.113694          0.703777          0.703777
4        currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s... 2022-03-20 13:05:00+08:00 2022-03-20 13:05:00+08:00                     1     high      1.221175    0.100343    0.000000          0.222879          0.222879
5        currencyservice2-0  container.node-6.container_cpu_cfs_throttled_p... 2022-03-20 13:05:00+08:00 2022-03-20 13:05:00+08:00                     1     high      1.000000    1.000000    0.000000          2.000000          2.000000
6               adservice-1  container.node-6.container_network_receive_MB.... 2022-03-20 13:05:00+08:00 2022-03-20 13:06:00+08:00                     2     high      4.065412    0.078176    0.056789          0.395991          0.380659
7         shippingservice-1  container.node-6.container_network_receive_MB.... 2022-03-20 13:06:00+08:00 2022-03-20 13:06:00+08:00                     1     high     11.459349    0.049917    0.034761          0.621930          0.621930
8         checkoutservice-1  container.node-6.container_cpu_cfs_throttled_s... 2022-03-20 13:07:00+08:00 2022-03-20 13:07:00+08:00                     1      low      1.000000    0.349237    0.005676          0.000000          0.000000
9         checkoutservice-1  container.node-6.container_network_receive_MB.... 2022-03-20 13:07:00+08:00 2022-03-20 13:07:00+08:00                     1     high      7.435951    0.077665    0.044869          0.655181          0.655181
10        checkoutservice-1  container.node-6.container_cpu_cfs_throttled_p... 2022-03-20 13:07:00+08:00 2022-03-20 13:07:00+08:00                     1      low      1.000000    4.000000    1.000000          0.000000          0.000000
11        currencyservice-1  container.node-6.container_network_receive_MB.... 2022-03-20 13:08:00+08:00 2022-03-20 13:08:00+08:00                     1     high      2.685937    0.203491    0.138716          0.750056          0.750056
12         paymentservice-2      container.node-6.container_cpu_system_seconds 2022-03-20 13:09:00+08:00 2022-03-20 13:09:00+08:00                     1     high      1.000000    0.005000    0.000000          0.010000          0.010000
13           emailservice-0  container.node-6.container_network_receive_MB.... 2022-03-20 13:09:00+08:00 2022-03-20 13:10:00+08:00                     2     high      8.523835    0.033480    0.022479          0.318860          0.311559
14        checkoutservice-0  container.node-6.container_network_receive_MB.... 2022-03-20 13:09:00+08:00 2022-03-20 13:10:00+08:00                     2    mixed      8.328975    0.069515    0.045265          0.648501          0.044914
15  recommendationservice-0  container.node-6.container_network_receive_MB.... 2022-03-20 13:10:00+08:00 2022-03-20 13:10:00+08:00                     1     high      2.670865    0.208148    0.158426          0.764081          0.764081
16        checkoutservice-1  container.node-6.container_cpu_cfs_throttled_s... 2022-03-20 13:10:00+08:00 2022-03-20 13:10:00+08:00                     1      low      0.746403    0.349237    0.005676          0.001439          0.001439
17         paymentservice-0  container.node-6.container_network_receive_pac... 2022-03-20 13:11:00+08:00 2022-03-20 13:11:00+08:00                     1     high      0.566494  289.500000  230.500000        453.500000        453.500000
18            cartservice-1  container.node-6.container_network_receive_MB.... 2022-03-20 13:11:00+08:00 2022-03-20 13:11:00+08:00                     1     high      5.435630    0.104238    0.069082          0.670837          0.670837
19         paymentservice-0  container.node-6.container_network_transmit_pa... 2022-03-20 13:11:00+08:00 2022-03-20 13:11:00+08:00                     1     high      0.720403  198.500000  152.000000        341.500000        341.500000```
```